> Federated Learning workflow with Autonomous client and model aggregation methos selection

### 1. Import Libraries

In [1]:
from abc import ABC, abstractmethod
from collections import deque, defaultdict
import copy
from dataclasses import dataclass, asdict, field
from enum import Enum
import json
import logging
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
from scipy import stats
from scipy.optimize import minimize
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
import tensorflow as tf
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset, Subset
from torchmetrics import Accuracy, Precision, Recall
import torchvision
from torchvision import datasets, transforms
from tqdm import tqdm
from typing import Dict, List, Optional, Tuple


import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

training_harware = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using training_harware: {training_harware}")

2025-09-17 09:58:06.229802: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758099486.497330  119232 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758099486.552453  119232 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758099486.991132  119232 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758099486.991216  119232 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758099486.991224  119232 computation_placer.cc:177] computation placer alr

Using training_harware: cpu


### 2. Model building

##### 2.1. TinyNet

In [2]:
class TinyNet(nn.Module):
    """Tiny CNN for resource-constrained IoT sensors (~5K parameters)"""
    
    def __init__(self, input_size: int, num_classes: int):
        super(TinyNet, self).__init__()
        self.input_size = input_size
        self.num_classes = num_classes
        
        # Reshape input to work with conv layers
        self.input_channels = min(8, input_size // 4)  # Adaptive channels
        self.conv_input_size = input_size // self.input_channels
        
        self.conv1 = nn.Conv1d(1, 4, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(4, 8, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(16)  # Fixed output size
        
        self.classifier = nn.Sequential(
            nn.Linear(8 * 16, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes)
        )
        
    def forward(self, x):
        # Handle both 1D (flattened) and 2D (image) inputs
        if len(x.shape) == 2:  # Flattened input
            x = x.unsqueeze(1)  # Add channel dimension
        elif len(x.shape) == 4:  # Image input (batch, height, width, channels)
            x = x.mean(dim=1, keepdim=True)  # Convert to grayscale if needed
            x = x.view(x.size(0), -1).unsqueeze(1)  # Flatten and add channel
        
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


##### 2.2. MobileNet

In [3]:
class MobileNet(nn.Module):
    """Mobile CNN for mid-range devices (~25K parameters)"""
    
    def __init__(self, input_size: int, num_classes: int):
        super(MobileNet, self).__init__()
        self.input_size = input_size
        self.num_classes = num_classes
        
        self.conv1 = nn.Conv1d(1, 16, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(32, 16, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(32)
        
        self.classifier = nn.Sequential(
            nn.Linear(16 * 32, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x):
        # Handle both 1D (flattened) and 2D (image) inputs
        if len(x.shape) == 2:  # Flattened input
            x = x.unsqueeze(1)  # Add channel dimension
        elif len(x.shape) == 4:  # Image input (batch, height, width, channels)
            x = x.mean(dim=1, keepdim=True)  # Convert to grayscale if needed
            x = x.view(x.size(0), -1).unsqueeze(1)  # Flatten and add channel
        
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


##### 2.3. StandardNet

In [4]:
class StandardNet(nn.Module):
    """Standard CNN for powerful gateway devices (~100K parameters)"""
    
    def __init__(self, input_size: int, num_classes: int):
        super(StandardNet, self).__init__()
        self.input_size = input_size
        self.num_classes = num_classes
        
        self.conv1 = nn.Conv1d(1, 32, kernel_size=7, padding=3)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(64, 32, kernel_size=3, padding=1)
        self.conv4 = nn.Conv1d(32, 16, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(64)
        
        self.classifier = nn.Sequential(
            nn.Linear(16 * 64, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x):
        # Handle both 1D (flattened) and 2D (image) inputs
        if len(x.shape) == 2:  # Flattened input
            x = x.unsqueeze(1)  # Add channel dimension
        elif len(x.shape) == 4:  # Image input (batch, height, width, channels)
            x = x.mean(dim=1, keepdim=True)  # Convert to grayscale if needed
            x = x.view(x.size(0), -1).unsqueeze(1)  # Flatten and add channel
        
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


#### 2.4. Dataset

In [5]:
class IDSDataset(Dataset):
    """Custom dataset for Intrusion Detection System data"""
    
    def __init__(self, features, labels, transform=None):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
        self.transform = transform
        
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        sample = self.features[idx]
        label = self.labels[idx]
        
        if self.transform:
            sample = self.transform(sample)
            
        return sample, label


In [6]:
def load_fashion_mnist_data() -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """Load Fashion-MNIST dataset"""
    
    # Download Fashion-MNIST
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    
    train_dataset = torchvision.datasets.FashionMNIST(root='./data', train=True, 
                                                      download=True, transform=transform)
    test_dataset = torchvision.datasets.FashionMNIST(root='./data', train=False, 
                                                     download=True, transform=transform)
    
    # Convert to numpy arrays
    X_train = train_dataset.data.numpy().reshape(60000, -1).astype(np.float32) / 255.0
    y_train = train_dataset.targets.numpy()
    
    X_test = test_dataset.data.numpy().reshape(10000, -1).astype(np.float32) / 255.0
    y_test = test_dataset.targets.numpy()
    
    # Combine train and test for redistribution
    X = np.vstack([X_train, X_test])
    y = np.hstack([y_train, y_test])
    
    class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
                   'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
    
    return X, y, class_names

def load_ids_dataset() -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """Load a real IDS dataset (using a public dataset as example)"""
    try:
        # Using NSL-KDD dataset as example (you can replace with Edge-IIoTset if available)
        # This is a placeholder - replace with actual dataset loading
        from sklearn.datasets import fetch_kddcup99
        
        print("Loading KDD Cup 99 dataset...")
        data = fetch_kddcup99(subset=None, data_home=None, download_if_missing=True, 
                             percent10=True, random_state=42)
        
        X = data.data
        y = data.target
        
        # Encode categorical features
        from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
        
        # Separate numeric and categorical columns
        categorical_columns = []
        numeric_columns = []
        
        for i in range(X.shape[1]):
            if isinstance(X[0, i], str):
                categorical_columns.append(i)
            else:
                numeric_columns.append(i)
        
        # Process categorical columns
        if categorical_columns:
            cat_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            X[:, categorical_columns] = cat_encoder.fit_transform(X[:, categorical_columns])
        
        # Convert to float
        X = X.astype(np.float32)
        
        # Encode labels
        label_encoder = LabelEncoder()
        y = label_encoder.fit_transform(y)
        
        # Get class names
        class_names = label_encoder.classes_.tolist()
        
        # Limit to reasonable size for testing
        if len(X) > 20000:
            indices = np.random.choice(len(X), 20000, replace=False)
            X = X[indices]
            y = y[indices]
        
        print(f"Loaded {len(class_names)} classes: {class_names[:10]}...")
        return X, y, class_names
        
    except Exception as e:
        print(f"Failed to load IDS dataset: {e}")
        print("Falling back to Fashion-MNIST...")
        return load_fashion_mnist_data()

def create_dirichlet_split(y: np.ndarray, n_clients: int, alpha: float = 0.5) -> List[np.ndarray]:
    """
    Create non-IID data splits using Dirichlet distribution
    
    Args:
        y: labels array
        n_clients: number of clients
        alpha: Dirichlet concentration parameter (lower = more non-IID)
    
    Returns:
        List of indices for each client
    """
    n_classes = len(np.unique(y))
    
    # Generate Dirichlet distribution for each client
    client_class_distributions = np.random.dirichlet([alpha] * n_classes, n_clients)
    
    # Get indices for each class
    class_indices = {}
    for class_id in range(n_classes):
        class_indices[class_id] = np.where(y == class_id)[0]
    
    # Distribute data to clients
    client_indices = [[] for _ in range(n_clients)]
    
    for class_id in range(n_classes):
        # Shuffle class indices
        np.random.shuffle(class_indices[class_id])
        
        # Calculate number of samples each client gets from this class
        class_size = len(class_indices[class_id])
        start_idx = 0
        
        for client_id in range(n_clients):
            # Calculate proportion for this client
            proportion = client_class_distributions[client_id, class_id]
            n_samples = int(proportion * class_size)
            
            # Ensure we don't exceed available samples
            end_idx = min(start_idx + n_samples, class_size)
            
            if start_idx < end_idx:
                client_indices[client_id].extend(class_indices[class_id][start_idx:end_idx])
                start_idx = end_idx
    
    # Convert to numpy arrays and shuffle
    for client_id in range(n_clients):
        client_indices[client_id] = np.array(client_indices[client_id])
        np.random.shuffle(client_indices[client_id])
    
    return client_indices

##### 2.5. Federated Learning settings

In [7]:
class FLModel:
    """Federated Learning Model wrapper"""
    
    def __init__(self, model_type: str, input_size: int, num_classes: int, device_capability: str):
        self.model_type = model_type
        self.input_size = input_size
        self.num_classes = num_classes
        self.device_capability = device_capability
        
        # Create model based on device capability
        if device_capability == "sensor":
            self.model = TinyNet(input_size, num_classes)
        elif device_capability == "mobile":
            self.model = MobileNet(input_size, num_classes)
        elif device_capability == "gateway":
            self.model = StandardNet(input_size, num_classes)
        else:
            self.model = MobileNet(input_size, num_classes)  # Default
            
        self.device = torch.device("cpu")  # For simplicity, using CPU
        self.model.to(self.device)
        
        # Training configuration
        self.optimizer = optim.Adam(self.model.parameters(), lr=0.001)
        self.criterion = nn.CrossEntropyLoss()
        self.scaler = StandardScaler()
        
        # Performance tracking
        self.training_history = []
        self.validation_history = []
        
    def get_model_size(self) -> int:
        """Get number of parameters in the model"""
        return sum(p.numel() for p in self.model.parameters() if p.requires_grad)
    
    def get_computational_complexity(self) -> float:
        """Estimate computational complexity (FLOPs approximation)"""
        model_size = self.get_model_size()
        complexity_factors = {
            "sensor": 1.0,
            "mobile": 2.5,
            "gateway": 5.0
        }
        return model_size * complexity_factors.get(self.device_capability, 2.0)
    
    def prepare_data(self, X: np.ndarray, y: np.ndarray, fit_scaler: bool = True) -> Tuple[IDSDataset, np.ndarray]:
        """Prepare data for training/testing"""
        if fit_scaler:
            X_scaled = self.scaler.fit_transform(X)
        else:
            X_scaled = self.scaler.transform(X)
            
        dataset = IDSDataset(X_scaled, y)
        return dataset, X_scaled
    
    def train_local(self, train_data: IDSDataset, val_data: Optional[IDSDataset] = None, 
                   epochs: int = 5, batch_size: int = 32) -> Dict[str, float]:
        """Train model locally on client data"""
        
        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
        self.model.train()
        
        epoch_losses = []
        epoch_accuracies = []
        
        for epoch in range(epochs):
            running_loss = 0.0
            correct_predictions = 0
            total_samples = 0
            
            for batch_features, batch_labels in train_loader:
                batch_features, batch_labels = batch_features.to(self.device), batch_labels.to(self.device)
                
                self.optimizer.zero_grad()
                outputs = self.model(batch_features)
                loss = self.criterion(outputs, batch_labels)
                loss.backward()
                self.optimizer.step()
                
                running_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total_samples += batch_labels.size(0)
                correct_predictions += (predicted == batch_labels).sum().item()
            
            epoch_loss = running_loss / len(train_loader)
            epoch_accuracy = correct_predictions / total_samples
            
            epoch_losses.append(epoch_loss)
            epoch_accuracies.append(epoch_accuracy)
        
        # Validation if provided
        val_metrics = {}
        if val_data is not None:
            val_metrics = self.evaluate(val_data)
        
        training_result = {
            'final_loss': epoch_losses[-1],
            'final_accuracy': epoch_accuracies[-1],
            'loss_history': epoch_losses,
            'accuracy_history': epoch_accuracies,
            'validation_metrics': val_metrics,
            'epochs_completed': epochs,
            'total_samples': len(train_data)
        }
        
        self.training_history.append(training_result)
        return training_result
    
    def evaluate(self, test_data: IDSDataset, batch_size: int = 64) -> Dict[str, float]:
        """Evaluate model performance with comprehensive metrics"""
        
        test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
        self.model.eval()
        
        all_predictions = []
        all_labels = []
        total_loss = 0.0
        
        with torch.no_grad():
            for batch_features, batch_labels in test_loader:
                batch_features, batch_labels = batch_features.to(self.device), batch_labels.to(self.device)
                
                outputs = self.model(batch_features)
                loss = self.criterion(outputs, batch_labels)
                total_loss += loss.item()
                
                _, predicted = torch.max(outputs, 1)
                all_predictions.extend(predicted.cpu().numpy())
                all_labels.extend(batch_labels.cpu().numpy())
        
        # Calculate comprehensive metrics
        accuracy = accuracy_score(all_labels, all_predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_predictions, average='macro', zero_division=0)
        precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(all_labels, all_predictions, average='micro', zero_division=0)
        
        # Per-class metrics
        precision_per_class, recall_per_class, f1_per_class, support = precision_recall_fscore_support(all_labels, all_predictions, average=None, zero_division=0)
        
        evaluation_result = {
            'loss': total_loss / len(test_loader),
            'accuracy': accuracy,
            'precision_macro': precision,
            'recall_macro': recall,
            'f1_macro': f1,
            'precision_micro': precision_micro,
            'recall_micro': recall_micro,
            'f1_micro': f1_micro,
            'per_class_precision': precision_per_class.tolist(),
            'per_class_recall': recall_per_class.tolist(),
            'per_class_f1': f1_per_class.tolist(),
            'per_class_support': support.tolist()
        }
        
        return evaluation_result
    
    def get_model_weights(self) -> Dict[str, np.ndarray]:
        """Get model weights for federated aggregation"""
        weights = {}
        for name, param in self.model.state_dict().items():
            weights[name] = param.cpu().numpy()
        return weights
    
    def set_model_weights(self, weights: Dict[str, np.ndarray]):
        """Set model weights from federated aggregation"""
        state_dict = {}
        for name, weight in weights.items():
            state_dict[name] = torch.FloatTensor(weight)
        self.model.load_state_dict(state_dict)

class FederatedAggregator:
    """Handles federated model aggregation"""
    
    def __init__(self, aggregation_method: str = "fedavg"):
        self.aggregation_method = aggregation_method
        self.aggregation_history = []
        
    def aggregate_models(self, client_models: List[Tuple[FLModel, Dict]], 
                        aggregation_weights: Optional[List[float]] = None) -> Dict[str, np.ndarray]:
        """Aggregate client models using specified method"""
        
        if not client_models:
            raise ValueError("No client models provided for aggregation")
        
        if aggregation_weights is None:
            # Default: weight by data size
            total_samples = sum(metadata.get('data_size', 1) for _, metadata in client_models)
            aggregation_weights = [metadata.get('data_size', 1) / total_samples for _, metadata in client_models]
        
        # Normalize weights
        total_weight = sum(aggregation_weights)
        aggregation_weights = [w / total_weight for w in aggregation_weights]
        
        if self.aggregation_method == "fedavg":
            return self._federated_averaging(client_models, aggregation_weights)
        elif self.aggregation_method == "weighted_avg":
            return self._weighted_averaging(client_models, aggregation_weights)
        elif self.aggregation_method == "size_aware":
            return self._size_aware_aggregation(client_models, aggregation_weights)
        else:
            return self._federated_averaging(client_models, aggregation_weights)
    
    def _federated_averaging(self, client_models: List[Tuple[FLModel, Dict]], 
                           weights: List[float]) -> Dict[str, np.ndarray]:
        """Standard FedAvg aggregation"""
        
        # Get all model weights
        all_weights = []
        for model, metadata in client_models:
            model_weights = model.get_model_weights()
            all_weights.append(model_weights)
        
        # Aggregate layer by layer
        aggregated_weights = {}
        
        # Get layer names from first model
        layer_names = list(all_weights[0].keys())
        
        for layer_name in layer_names:
            # Check if all models have this layer (for heterogeneous models)
            layer_weights = []
            layer_client_weights = []
            
            for i, model_weights in enumerate(all_weights):
                if layer_name in model_weights:
                    layer_weights.append(model_weights[layer_name])
                    layer_client_weights.append(weights[i])
            
            if layer_weights:
                # Normalize weights for this layer
                total_layer_weight = sum(layer_client_weights)
                normalized_layer_weights = [w / total_layer_weight for w in layer_client_weights]
                
                # Weighted average
                aggregated_layer = np.zeros_like(layer_weights[0])
                for weight, layer_weight in zip(normalized_layer_weights, layer_weights):
                    aggregated_layer += weight * layer_weight
                
                aggregated_weights[layer_name] = aggregated_layer
        
        # Record aggregation info
        self.aggregation_history.append({
            'method': 'fedavg',
            'num_clients': len(client_models),
            'weights_used': weights,
            'layers_aggregated': len(aggregated_weights)
        })
        
        return aggregated_weights
    
    def _weighted_averaging(self, client_models: List[Tuple[FLModel, Dict]], 
                          weights: List[float]) -> Dict[str, np.ndarray]:
        """Weighted averaging based on model performance"""
        
        # Adjust weights based on model performance
        performance_weights = []
        for model, metadata in client_models:
            # Use validation F1 score if available, otherwise use training accuracy
            if 'validation_f1' in metadata:
                perf_weight = metadata['validation_f1']
            elif 'training_accuracy' in metadata:
                perf_weight = metadata['training_accuracy']
            else:
                perf_weight = 1.0
            performance_weights.append(perf_weight)
        
        # Combine data size weights and performance weights
        combined_weights = []
        for i in range(len(weights)):
            combined_weight = weights[i] * (0.7 + 0.3 * performance_weights[i])  # 70% data size, 30% performance
            combined_weights.append(combined_weight)
        
        # Normalize
        total_weight = sum(combined_weights)
        combined_weights = [w / total_weight for w in combined_weights]
        
        return self._federated_averaging(client_models, combined_weights)
    
    def _size_aware_aggregation(self, client_models: List[Tuple[FLModel, Dict]], 
                               weights: List[float]) -> Dict[str, np.ndarray]:
        """Size-aware aggregation for heterogeneous model sizes"""
        
        # Group models by size/capability
        size_groups = defaultdict(list)
        for i, (model, metadata) in enumerate(client_models):
            capability = model.device_capability
            size_groups[capability].append((i, model, metadata, weights[i]))
        
        # Aggregate within each size group first
        group_aggregates = {}
        group_weights = {}
        
        for capability, group_models in size_groups.items():
            if len(group_models) == 1:
                # Single model in group
                _, model, metadata, weight = group_models[0]
                group_aggregates[capability] = model.get_model_weights()
                group_weights[capability] = weight
            else:
                # Multiple models in group - use standard FedAvg
                group_client_models = [(model, metadata) for _, model, metadata, _ in group_models]
                group_client_weights = [weight for _, _, _, weight in group_models]
                
                # Normalize group weights
                total_group_weight = sum(group_client_weights)
                normalized_group_weights = [w / total_group_weight for w in group_client_weights]
                
                group_aggregates[capability] = self._federated_averaging(group_client_models, normalized_group_weights)
                group_weights[capability] = total_group_weight
        
        # Now aggregate across groups (use largest model as template)
        largest_capability = max(group_weights.keys(), key=lambda x: {'sensor': 1, 'mobile': 2, 'gateway': 3}[x])
        final_aggregate = group_aggregates[largest_capability].copy()
        
        # Record aggregation info
        self.aggregation_history.append({
            'method': 'size_aware',
            'num_clients': len(client_models),
            'size_groups': {k: len(v) for k, v in size_groups.items()},
            'largest_model': largest_capability
        })
        
        return final_aggregate


#### 3. IoT device properties

##### 3.1. Resource information model

In [8]:
class DeviceStatus(Enum):
    ACTIVE = "active"
    IDLE = "idle"
    FAILED = "failed"
    SLEEPING = "sleeping"

@dataclass
class DeviceSpecs:
    """Static device specifications"""
    max_battery: float  # mAh
    max_cpu_freq: float  # GHz
    max_bandwidth_up: float  # Mbps
    max_bandwidth_down: float  # Mbps
    memory: float  # GB
    device_type: str  # "sensor", "gateway", "mobile", etc.

@dataclass
class DeviceState:
    """Dynamic device state"""
    current_battery: float  # Current battery level (0-1)
    current_cpu_load: float  # Current CPU utilization (0-1)
    current_bandwidth_up: float  # Available uplink bandwidth
    current_bandwidth_down: float  # Available downlink bandwidth
    temperature: float  # Device temperature (Celsius)
    status: DeviceStatus
    last_update: float  # Timestamp

@dataclass
class ResourceConsumption:
    """Resource consumption for different FL operations"""
    computation_energy: float  # mAh per training operation
    communication_energy_tx: float  # mAh per MB transmitted
    communication_energy_rx: float  # mAh per MB received
    idle_energy: float  # mAh per second in idle state
    base_energy: float  # Base energy consumption per second

class BatteryModel:
    """Advanced battery discharge modeling"""
    
    def __init__(self, capacity_mah: float, initial_charge: float = 1.0):
        self.capacity_mah = capacity_mah
        self.current_charge = initial_charge  # 0-1
        self.discharge_history = []
        self.temperature_factor = 1.0
        self.age_factor = 1.0  # Battery degradation over time
        
    def calculate_discharge_rate(self, power_consumption: float, temperature: float) -> float:
        """Calculate battery discharge rate considering non-linear effects"""
        # Temperature effect: batteries perform worse in extreme temperatures
        temp_efficiency = self._get_temperature_efficiency(temperature)
        
        # Non-linear discharge: higher power consumption leads to lower efficiency
        power_efficiency = self._get_power_efficiency(power_consumption)
        
        # Peukert's law approximation for battery capacity vs discharge rate
        peukert_factor = 1.0 + (power_consumption / self.capacity_mah) * 0.3
        
        actual_consumption = power_consumption * peukert_factor / (temp_efficiency * power_efficiency * self.age_factor)
        
        return actual_consumption / self.capacity_mah  # Return as fraction of total capacity
    
    def _get_temperature_efficiency(self, temperature: float) -> float:
        """Battery efficiency based on temperature (optimal around 20-25°C)"""
        optimal_temp = 22.5
        temp_diff = abs(temperature - optimal_temp)
        return max(0.5, 1.0 - (temp_diff / 50.0))  # Efficiency drops with temperature deviation
    
    def _get_power_efficiency(self, power_mah: float) -> float:
        """Battery efficiency decreases with higher power draw"""
        normalized_power = power_mah / self.capacity_mah
        return max(0.6, 1.0 - normalized_power * 0.4)
    
    def discharge(self, energy_consumption_mah: float, temperature: float, time_seconds: float = 1.0) -> bool:
        """Discharge battery and return True if still alive"""
        if self.current_charge <= 0:
            return False
        
        discharge_rate = self.calculate_discharge_rate(energy_consumption_mah, temperature)
        discharge_amount = discharge_rate * (time_seconds / 3600.0)  # Convert to hours
        
        self.current_charge = max(0, self.current_charge - discharge_amount)
        self.discharge_history.append({
            'timestamp': time.time(),
            'consumption': energy_consumption_mah,
            'charge_after': self.current_charge,
            'temperature': temperature
        })
        
        return self.current_charge > 0
    
    def get_remaining_capacity(self) -> float:
        """Get remaining battery capacity in mAh"""
        return self.current_charge * self.capacity_mah
    
    def get_estimated_lifetime(self, avg_power_consumption: float, temperature: float = 25.0) -> float:
        """Estimate remaining lifetime in hours given average power consumption"""
        if avg_power_consumption <= 0 or self.current_charge <= 0:
            return 0
        
        discharge_rate = self.calculate_discharge_rate(avg_power_consumption, temperature)
        return self.current_charge / discharge_rate if discharge_rate > 0 else float('inf')

class BandwidthPredictor:
    """Predict available bandwidth based on historical patterns"""
    
    def __init__(self, max_bandwidth: float, base_bandwidth: float):
        self.max_bandwidth = max_bandwidth
        self.base_bandwidth = base_bandwidth
        self.bandwidth_history = []
        self.interference_level = 0.1  # 0-1, network congestion factor
        
    def predict_bandwidth(self, current_time: float, time_of_day_factor: bool = True) -> Tuple[float, float]:
        """Predict uplink and downlink bandwidth"""
        # Time-of-day pattern (lower bandwidth during peak hours)
        time_factor = 1.0
        if time_of_day_factor:
            hour = (current_time % 86400) / 3600  # Get hour of day
            # Peak hours: 8-10 AM, 12-2 PM, 6-10 PM
            peak_hours = [(8, 10), (12, 14), (18, 22)]
            for start, end in peak_hours:
                if start <= hour <= end:
                    time_factor = 0.6 + 0.4 * np.random.random()  # 60-100% during peak
                    break
        
        # Interference and congestion effects
        interference_factor = 1.0 - self.interference_level * np.random.random()
        
        # Random variations
        random_factor = 0.8 + 0.4 * np.random.random()  # ±20% variation
        
        predicted_bandwidth = self.max_bandwidth * time_factor * interference_factor * random_factor
        predicted_bandwidth = max(self.base_bandwidth, predicted_bandwidth)
        
        # Uplink typically lower than downlink for most IoT devices
        uplink_ratio = 0.3 + 0.4 * np.random.random()  # 30-70% of downlink
        
        uplink = predicted_bandwidth * uplink_ratio
        downlink = predicted_bandwidth
        
        self.bandwidth_history.append({
            'timestamp': current_time,
            'uplink': uplink,
            'downlink': downlink,
            'time_factor': time_factor,
            'interference_factor': interference_factor
        })
        
        return uplink, downlink
    
    def update_interference(self, new_interference: float):
        """Update network interference level"""
        self.interference_level = max(0, min(1, new_interference))

class CPUModel:
    """Model CPU performance with thermal throttling and load effects"""
    
    def __init__(self, max_frequency: float, base_temperature: float = 35.0):
        self.max_frequency = max_frequency  # GHz
        self.base_temperature = base_temperature
        self.current_load = 0.1  # Current CPU utilization (0-1)
        self.current_temperature = base_temperature
        self.thermal_threshold = 70.0  # Start throttling above this temperature
        self.performance_history = []
        
    def calculate_effective_performance(self, workload: float, ambient_temp: float = 25.0) -> Dict[str, float]:
        """Calculate effective CPU performance considering thermal throttling"""
        # Update temperature based on load and ambient conditions
        self.current_temperature = self._calculate_temperature(workload, ambient_temp)
        
        # Thermal throttling factor
        thermal_factor = self._get_thermal_throttling_factor()
        
        # Load-based performance (efficiency decreases at very high loads)
        load_factor = self._get_load_efficiency_factor(workload)
        
        # Calculate effective frequency and performance
        effective_frequency = self.max_frequency * thermal_factor * load_factor
        
        # Performance metrics
        performance_metrics = {
            'effective_frequency': effective_frequency,
            'thermal_factor': thermal_factor,
            'load_factor': load_factor,
            'temperature': self.current_temperature,
            'power_consumption': self._calculate_power_consumption(effective_frequency, workload)
        }
        
        self.performance_history.append({
            'timestamp': time.time(),
            'workload': workload,
            **performance_metrics
        })
        
        return performance_metrics
    
    def _calculate_temperature(self, cpu_load: float, ambient_temp: float) -> float:
        """Calculate CPU temperature based on load and ambient conditions"""
        # Simplified thermal model: temperature rises with CPU load
        load_heat = cpu_load * 35.0  # Max 35°C increase under full load
        thermal_inertia = 0.8  # How quickly temperature changes
        
        target_temp = ambient_temp + self.base_temperature - ambient_temp + load_heat
        self.current_temperature = (thermal_inertia * self.current_temperature + 
                                  (1 - thermal_inertia) * target_temp)
        
        return self.current_temperature
    
    def _get_thermal_throttling_factor(self) -> float:
        """Calculate performance reduction due to thermal throttling"""
        if self.current_temperature <= self.thermal_threshold:
            return 1.0
        
        excess_temp = self.current_temperature - self.thermal_threshold
        # Linear throttling: 10% reduction per 10°C above threshold
        throttling = max(0.3, 1.0 - (excess_temp / 100.0))
        return throttling
    
    def _get_load_efficiency_factor(self, load: float) -> float:
        """CPU efficiency decreases at very high loads due to context switching overhead"""
        if load <= 0.7:
            return 1.0
        else:
            # Efficiency drops after 70% load
            return max(0.7, 1.0 - (load - 0.7) * 0.5)
    
    def _calculate_power_consumption(self, frequency: float, load: float) -> float:
        """Calculate CPU power consumption in watts"""
        # Simplified power model: P = C * V² * f * α
        # Where C is capacitance, V is voltage, f is frequency, α is activity factor
        base_power = 0.5  # Base power consumption in watts
        dynamic_power = (frequency / self.max_frequency) ** 2 * load * 2.0  # Dynamic power
        return base_power + dynamic_power

class ResourceManager:
    """Manages all dynamic resource models for an IoT client"""
    
    def __init__(self, client_specs, consumption_profile: ResourceConsumption):
        self.battery_model = BatteryModel(client_specs.max_battery)
        self.bandwidth_predictor = BandwidthPredictor(
            client_specs.max_bandwidth_up, 
            client_specs.max_bandwidth_up * 0.1  # 10% minimum
        )
        self.cpu_model = CPUModel(client_specs.max_cpu_freq)
        self.consumption_profile = consumption_profile
        
        # Environmental factors
        self.ambient_temperature = 25.0
        self.network_interference = 0.1
        
    def update_resources(self, fl_operation: str = "idle", data_size_mb: float = 0, 
                        duration_seconds: float = 1.0) -> Dict[str, float]:
        """Update all resource models and return current state"""
        current_time = time.time()
        
        # Calculate energy consumption based on operation
        energy_consumption = self._calculate_energy_consumption(fl_operation, data_size_mb, duration_seconds)
        
        # Update battery
        battery_alive = self.battery_model.discharge(
            energy_consumption, 
            self.cpu_model.current_temperature, 
            duration_seconds
        )
        
        # Update bandwidth prediction
        uplink, downlink = self.bandwidth_predictor.predict_bandwidth(current_time)
        
        # Update CPU performance
        cpu_load = self._get_cpu_load_for_operation(fl_operation)
        cpu_performance = self.cpu_model.calculate_effective_performance(cpu_load, self.ambient_temperature)
        
        return {
            'battery_level': self.battery_model.current_charge,
            'battery_alive': battery_alive,
            'bandwidth_up': uplink,
            'bandwidth_down': downlink,
            'cpu_performance': cpu_performance,
            'energy_consumed': energy_consumption,
            'estimated_lifetime_hours': self.battery_model.get_estimated_lifetime(
                self.consumption_profile.base_energy, self.cpu_model.current_temperature
            )
        }
    
    def _calculate_energy_consumption(self, operation: str, data_size_mb: float, duration_seconds: float) -> float:
        """Calculate energy consumption for specific FL operation"""
        base_consumption = self.consumption_profile.base_energy * duration_seconds
        
        if operation == "training":
            computation_energy = self.consumption_profile.computation_energy
            comm_energy = (data_size_mb * self.consumption_profile.communication_energy_tx)  # Upload model updates
            return base_consumption + computation_energy + comm_energy
        
        elif operation == "model_download":
            comm_energy = data_size_mb * self.consumption_profile.communication_energy_rx
            return base_consumption + comm_energy
        
        elif operation == "idle":
            return base_consumption + self.consumption_profile.idle_energy * duration_seconds
        
        return base_consumption
    
    def _get_cpu_load_for_operation(self, operation: str) -> float:
        """Get CPU load for different FL operations"""
        load_mapping = {
            "training": 0.8 + 0.2 * np.random.random(),  # 80-100% during training
            "model_download": 0.3 + 0.2 * np.random.random(),  # 30-50% during download
            "idle": 0.05 + 0.1 * np.random.random(),  # 5-15% idle load
            "communication": 0.2 + 0.3 * np.random.random()  # 20-50% during communication
        }
        return load_mapping.get(operation, 0.1)
    
    def set_environmental_conditions(self, ambient_temp: float, interference: float):
        """Set environmental conditions affecting resource consumption"""
        self.ambient_temperature = ambient_temp
        self.network_interference = interference
        self.bandwidth_predictor.update_interference(interference)


##### 3.2. Carbon footprint

In [9]:
class EnergySource(Enum):
    """Different energy sources with varying carbon intensities"""
    GRID_AVERAGE = "grid_average"
    COAL = "coal"
    NATURAL_GAS = "natural_gas" 
    RENEWABLE = "renewable"
    SOLAR = "solar"
    WIND = "wind"
    NUCLEAR = "nuclear"
    BATTERY = "battery"  # For battery-powered devices

@dataclass
class CarbonIntensityData:
    """Carbon intensity values for different energy sources and operations"""
    # Carbon intensity in gCO2eq/kWh
    grid_average: float = 475.0  # Global average
    coal: float = 820.0
    natural_gas: float = 350.0
    renewable: float = 40.0
    solar: float = 48.0
    wind: float = 11.0
    nuclear: float = 12.0
    
    # Network infrastructure carbon intensity (gCO2eq/GB)
    network_transmission: float = 4.6  # Data transmission
    data_center_processing: float = 7.2  # Server processing
    
    # Manufacturing embodied carbon (gCO2eq per device)
    iot_device_embodied: float = 50000.0  # Small IoT sensor
    mobile_device_embodied: float = 85000.0  # Smartphone/tablet
    gateway_device_embodied: float = 150000.0  # Network gateway
    
    @classmethod
    def get_regional_intensity(cls, region: str = "global") -> float:
        """Get regional carbon intensity values"""
        regional_values = {
            "global": 475.0,
            "europe": 296.0,
            "usa": 386.0,
            "china": 681.0,
            "india": 709.0,
            "africa": 650.0,
            "nordic": 83.0,  # High renewable penetration
            "coal_heavy": 800.0  # Coal-dependent regions
        }
        return regional_values.get(region, 475.0)

class CarbonFootprintTracker:
    """Track carbon footprint for individual clients and system-wide"""
    
    def __init__(self, client_id: int, device_type: str, region: str = "global"):
        self.client_id = client_id
        self.device_type = device_type
        self.region = region
        
        # Carbon intensity data
        self.carbon_data = CarbonIntensityData()
        self.grid_intensity = self.carbon_data.get_regional_intensity(region)
        
        # Tracking variables
        self.total_energy_consumed_kwh = 0.0
        self.total_data_transmitted_gb = 0.0
        self.total_carbon_emissions_g = 0.0
        
        # Detailed tracking
        self.emissions_by_operation = {
            'computation': 0.0,
            'communication': 0.0,
            'idle': 0.0,
            'embodied': 0.0
        }
        
        self.emissions_history = []
        
        # Device-specific factors
        self.device_efficiency_factor = self._get_device_efficiency_factor()
        self.embodied_carbon_per_hour = self._calculate_embodied_carbon_rate()
        
    def _get_device_efficiency_factor(self) -> float:
        """Get device-specific energy efficiency factor"""
        efficiency_factors = {
            "sensor": 1.0,      # Baseline efficiency
            "mobile": 0.8,      # More efficient processors
            "gateway": 1.2,     # Less efficient, more powerful
            "server": 1.5       # Least efficient per operation
        }
        return efficiency_factors.get(self.device_type, 1.0)
    
    def _calculate_embodied_carbon_rate(self) -> float:
        """Calculate embodied carbon emissions per hour of operation"""
        # Assume device lifetime of 3-5 years depending on type
        device_lifetimes_hours = {
            "sensor": 3 * 365 * 24,    # 3 years
            "mobile": 4 * 365 * 24,    # 4 years
            "gateway": 5 * 365 * 24,   # 5 years
        }
        
        embodied_carbon_total = {
            "sensor": self.carbon_data.iot_device_embodied,
            "mobile": self.carbon_data.mobile_device_embodied,
            "gateway": self.carbon_data.gateway_device_embodied,
        }
        
        lifetime_hours = device_lifetimes_hours.get(self.device_type, 4 * 365 * 24)
        total_embodied = embodied_carbon_total.get(self.device_type, self.carbon_data.iot_device_embodied)
        
        return total_embodied / lifetime_hours  # gCO2eq per hour
    
    def calculate_operation_carbon(self, operation: str, energy_consumed_mah: float, 
                                 data_size_mb: float = 0, duration_seconds: float = 1.0) -> Dict[str, float]:
        """Calculate carbon footprint for a specific FL operation"""
        
        # Convert mAh to kWh (assuming 3.7V average for Li-ion batteries)
        voltage = 3.7  # Volts
        energy_kwh = (energy_consumed_mah * voltage) / (1000 * 1000)  # Convert mAh*V to kWh
        
        # Apply device efficiency factor
        effective_energy_kwh = energy_kwh * self.device_efficiency_factor
        
        # Calculate carbon emissions for different components
        carbon_breakdown = {}
        
        # 1. Computational carbon (direct energy consumption)
        if operation in ["training", "inference"]:
            computation_carbon = effective_energy_kwh * self.grid_intensity
            carbon_breakdown['computation'] = computation_carbon
        else:
            carbon_breakdown['computation'] = 0.0
        
        # 2. Communication carbon (data transmission + network infrastructure)
        if data_size_mb > 0:
            data_gb = data_size_mb / 1000.0
            
            # Network transmission carbon
            transmission_carbon = data_gb * self.carbon_data.network_transmission
            
            # Data center processing carbon (for model aggregation)
            if operation in ["model_upload", "training"]:
                processing_carbon = data_gb * self.carbon_data.data_center_processing
            else:
                processing_carbon = 0.0
            
            carbon_breakdown['communication'] = transmission_carbon + processing_carbon
        else:
            carbon_breakdown['communication'] = 0.0
        
        # 3. Idle/baseline carbon
        if operation == "idle":
            idle_carbon = effective_energy_kwh * self.grid_intensity
            carbon_breakdown['idle'] = idle_carbon
        else:
            # Include baseline consumption during active operations
            baseline_ratio = 0.2  # 20% of energy is baseline consumption
            baseline_carbon = effective_energy_kwh * baseline_ratio * self.grid_intensity
            carbon_breakdown['idle'] = baseline_carbon
        
        # 4. Embodied carbon (amortized per hour)
        duration_hours = duration_seconds / 3600.0
        carbon_breakdown['embodied'] = self.embodied_carbon_per_hour * duration_hours
        
        # Total carbon for this operation
        total_carbon = sum(carbon_breakdown.values())
        
        # Update tracking
        self.total_energy_consumed_kwh += effective_energy_kwh
        self.total_data_transmitted_gb += data_size_mb / 1000.0
        self.total_carbon_emissions_g += total_carbon
        
        # Update by operation type
        for category, value in carbon_breakdown.items():
            self.emissions_by_operation[category] += value
        
        # Record in history
        self.emissions_history.append({
            'timestamp': time.time(),
            'operation': operation,
            'energy_kwh': effective_energy_kwh,
            'data_mb': data_size_mb,
            'duration_s': duration_seconds,
            'carbon_g': total_carbon,
            'breakdown': carbon_breakdown.copy()
        })
        
        return {
            'total_carbon_g': total_carbon,
            'carbon_breakdown': carbon_breakdown,
            'energy_efficiency_gco2_per_kwh': total_carbon / max(effective_energy_kwh, 0.0001),
            'carbon_intensity_gco2_per_mb': total_carbon / max(data_size_mb, 0.001) if data_size_mb > 0 else 0
        }
    
    def get_carbon_summary(self) -> Dict[str, float]:
        """Get comprehensive carbon footprint summary"""
        return {
            'total_carbon_emissions_g': self.total_carbon_emissions_g,
            'total_energy_consumed_kwh': self.total_energy_consumed_kwh,
            'total_data_transmitted_gb': self.total_data_transmitted_gb,
            'carbon_intensity_avg': self.total_carbon_emissions_g / max(self.total_energy_consumed_kwh, 0.0001),
            'emissions_by_operation': self.emissions_by_operation.copy(),
            'carbon_per_gb': self.total_carbon_emissions_g / max(self.total_data_transmitted_gb, 0.001),
            'embodied_carbon_percentage': (self.emissions_by_operation['embodied'] / max(self.total_carbon_emissions_g, 0.001)) * 100
        }

class SystemCarbonFootprint:
    """Track system-wide carbon footprint across all clients and infrastructure"""
    
    def __init__(self, region: str = "global"):
        self.region = region
        self.carbon_data = CarbonIntensityData()
        self.client_trackers: Dict[int, CarbonFootprintTracker] = {}
        
        # System-wide tracking
        self.server_carbon_tracker = None
        self.gateway_carbon_trackers = {}
        
        # Aggregated metrics
        self.system_totals = {
            'total_carbon_g': 0.0,
            'total_energy_kwh': 0.0,
            'total_data_gb': 0.0,
            'training_rounds': 0,
            'active_clients': 0
        }
        
        self.round_history = []
        
    def add_client_tracker(self, client_id: int, device_type: str):
        """Add carbon tracking for a new client"""
        self.client_trackers[client_id] = CarbonFootprintTracker(client_id, device_type, self.region)
    
    def record_client_operation(self, client_id: int, operation: str, energy_mah: float,
                               data_mb: float = 0, duration_s: float = 1.0) -> Dict[str, float]:
        """Record carbon footprint for a client operation"""
        if client_id not in self.client_trackers:
            # Auto-create tracker with default device type
            self.add_client_tracker(client_id, "sensor")
        
        return self.client_trackers[client_id].calculate_operation_carbon(
            operation, energy_mah, data_mb, duration_s
        )
    
    def record_training_round(self, round_num: int, participating_clients: List[int],
                            server_energy_kwh: float = 0.1) -> Dict[str, float]:
        """Record carbon footprint for a complete FL training round"""
        
        # Calculate client-side carbon
        round_carbon = 0.0
        round_energy = 0.0
        
        for client_id in participating_clients:
            if client_id in self.client_trackers:
                tracker = self.client_trackers[client_id]
                # Get recent carbon emissions (last operation)
                if tracker.emissions_history:
                    latest = tracker.emissions_history[-1]
                    round_carbon += latest['carbon_g']
                    round_energy += latest['energy_kwh']
        
        # Add server-side carbon (model aggregation)
        server_carbon = server_energy_kwh * self.carbon_data.get_regional_intensity(self.region)
        round_carbon += server_carbon
        round_energy += server_energy_kwh
        
        # Update system totals
        self.system_totals['total_carbon_g'] += round_carbon
        self.system_totals['total_energy_kwh'] += round_energy
        self.system_totals['training_rounds'] += 1
        self.system_totals['active_clients'] = len(participating_clients)
        
        # Record round history
        round_metrics = {
            'round_num': round_num,
            'timestamp': time.time(),
            'participating_clients': len(participating_clients),
            'round_carbon_g': round_carbon,
            'round_energy_kwh': round_energy,
            'server_carbon_g': server_carbon,
            'carbon_per_client': round_carbon / max(len(participating_clients), 1),
            'carbon_efficiency': round_carbon / max(round_energy, 0.0001)  # gCO2/kWh
        }
        
        self.round_history.append(round_metrics)
        
        return round_metrics
    
    def get_system_carbon_summary(self) -> Dict[str, any]:
        """Get comprehensive system-wide carbon footprint analysis"""
        
        # Aggregate client statistics
        client_summaries = {}
        total_client_carbon = 0.0
        
        for client_id, tracker in self.client_trackers.items():
            summary = tracker.get_carbon_summary()
            client_summaries[client_id] = summary
            total_client_carbon += summary['total_carbon_emissions_g']
        
        # Calculate system-wide metrics
        avg_round_carbon = np.mean([r['round_carbon_g'] for r in self.round_history]) if self.round_history else 0
        total_rounds = len(self.round_history)
        
        # Carbon efficiency metrics
        carbon_per_round = self.system_totals['total_carbon_g'] / max(total_rounds, 1)
        carbon_per_client_round = carbon_per_round / max(self.system_totals['active_clients'], 1)
        
        # Breakdown analysis
        operation_breakdown = {'computation': 0, 'communication': 0, 'idle': 0, 'embodied': 0}
        for tracker in self.client_trackers.values():
            for op, value in tracker.emissions_by_operation.items():
                operation_breakdown[op] += value
        
        return {
            'system_totals': self.system_totals.copy(),
            'carbon_efficiency_metrics': {
                'carbon_per_round_g': carbon_per_round,
                'carbon_per_client_round_g': carbon_per_client_round,
                'carbon_intensity_gco2_per_kwh': self.system_totals['total_carbon_g'] / max(self.system_totals['total_energy_kwh'], 0.0001),
                'avg_round_carbon_g': avg_round_carbon
            },
            'operation_breakdown': operation_breakdown,
            'operation_breakdown_percentage': {
                op: (value / max(self.system_totals['total_carbon_g'], 0.001)) * 100 
                for op, value in operation_breakdown.items()
            },
            'client_count': len(self.client_trackers),
            'total_rounds': total_rounds,
            'region': self.region,
            'carbon_intensity_region': self.carbon_data.get_regional_intensity(self.region)
        }
    
    def compare_carbon_scenarios(self, scenarios: List[str]) -> Dict[str, Dict]:
        """Compare carbon footprint under different scenarios"""
        comparisons = {}
        
        for scenario in scenarios:
            # Create temporary system with different settings
            scenario_carbon = self.carbon_data.get_regional_intensity(scenario.lower())
            scenario_reduction = (self.carbon_data.get_regional_intensity(self.region) - scenario_carbon) / self.carbon_data.get_regional_intensity(self.region)
            
            comparisons[scenario] = {
                'carbon_intensity': scenario_carbon,
                'total_carbon_reduction_g': self.system_totals['total_carbon_g'] * scenario_reduction,
                'percentage_reduction': scenario_reduction * 100,
                'equivalent_total_g': self.system_totals['total_carbon_g'] * (1 - scenario_reduction)
            }
        
        return comparisons
    
    def export_carbon_data(self) -> Dict:
        """Export all carbon tracking data for analysis"""
        return {
            'system_summary': self.get_system_carbon_summary(),
            'round_history': self.round_history,
            'client_trackers': {
                client_id: {
                    'summary': tracker.get_carbon_summary(),
                    'history': tracker.emissions_history[-10:]  # Last 10 operations
                }
                for client_id, tracker in self.client_trackers.items()
            }
        }


#### 4. IoI Client

In [10]:
class IoTClient:
    """Represents an IoT device in the federated learning network"""
    
    def __init__(self, client_id: int, specs: DeviceSpecs, initial_position: Tuple[float, float] = (0, 0)):
        self.client_id = client_id
        self.specs = specs
        self.state = DeviceState(
            current_battery=1.0,  # Start with full battery
            current_cpu_load=0.1,  # Minimal base load
            current_bandwidth_up=specs.max_bandwidth_up * 0.8,  # 80% of max initially
            current_bandwidth_down=specs.max_bandwidth_down * 0.8,
            temperature=25.0,  # Room temperature
            status=DeviceStatus.ACTIVE,
            last_update=time.time()
        )
        
        # FL-specific attributes
        self.local_data_size = 0
        self.data_quality_score = 1.0  # 0-1, higher is better
        self.gradient_quality_score = 1.0  # 0-1, based on data quality and local training
        
        # Position for mobility modeling (if needed)
        self.position = initial_position

        self.carbon_tracker = CarbonFootprintTracker(client_id, specs.device_type, "africa")
        
        # Training history
        self.participation_history = []
        self.energy_consumption_history = []
        self.carbon_footprint_history = []
        
        # Create resource consumption profile based on device type
        consumption_profiles = {
            "sensor": ResourceConsumption(30.0, 1.5, 1.0, 0.05, 0.3),
            "gateway": ResourceConsumption(80.0, 3.0, 2.0, 0.2, 0.8), 
            "mobile": ResourceConsumption(60.0, 2.5, 1.8, 0.1, 0.5)
        }
        default_consumption = ResourceConsumption(50.0, 2.0, 1.5, 0.1, 0.5)

        # Initialize resource manager
        self.resource_manager = ResourceManager(
            specs, 
            consumption_profiles.get(specs.device_type, default_consumption)
        )

        self.fl_model = None  # Will be initialized when FL system starts
        self.local_dataset = None
        self.model_performance_history = []
        self.current_model_accuracy = 0.0
        self.current_model_f1 = 0.0
        
        # Update gradient quality based on device capability
        device_capability_factors = {
            "sensor": 0.7,
            "mobile": 0.85,
            "gateway": 1.0
        }
        base_factor = device_capability_factors.get(specs.device_type, 0.8)
        self.gradient_quality_score = self.data_quality_score * base_factor

    def initialize_fl_model(self, model_type: str, input_size: int, num_classes: int):
        """Initialize the FL model for this client"""
        device_capability = self.specs.device_type
        self.fl_model = FLModel(model_type, input_size, num_classes, device_capability)
    
    def set_local_data(self, X_local: np.ndarray, y_local: np.ndarray):
        """Set local training data for this client"""
        if self.fl_model is not None:
            self.local_dataset, _ = self.fl_model.prepare_data(X_local, y_local, fit_scaler=True)
            self.local_data_size = len(X_local)
            self._local_y = y_local  # Store for later data quality calculation
            
            # Calculate data quality based on class distribution and size
            unique_classes = len(np.unique(y_local))
            data_size_factor = min(1.0, len(X_local) / 1000.0)
            class_diversity_factor = min(1.0, unique_classes / 10.0)  # Temporary, will be updated
            
            self.data_quality_score = min(1.0, (data_size_factor * 0.6 + class_diversity_factor * 0.4))
            
            # Update gradient quality
            device_capability_factors = {
                "sensor": 0.7,
                "mobile": 0.85,
                "gateway": 1.0
            }
            base_factor = device_capability_factors.get(self.specs.device_type, 0.8)
            self.gradient_quality_score = self.data_quality_score * base_factor
            
    def train_local_model(self, epochs: int = 3, batch_size: int = 32) -> Dict[str, any]:
        """Train the local FL model"""
        if self.fl_model is None or self.local_dataset is None:
            return {'error': 'FL model or local data not initialized'}
        
        # Simulate training energy consumption
        model_complexity = self.fl_model.get_computational_complexity()
        training_energy_mah = (model_complexity / 1000000) * epochs * 20  # Rough estimation
        
        # Update dynamic state for training
        training_state = self.update_dynamic_state("training", 0, epochs * 30)  # 30 sec per epoch
        
        # Perform actual training
        training_result = self.fl_model.train_local(
            self.local_dataset, 
            epochs=epochs, 
            batch_size=batch_size
        )
        
        # Update client performance metrics
        self.current_model_accuracy = training_result['final_accuracy']
        if 'validation_metrics' in training_result and 'f1_macro' in training_result['validation_metrics']:
            self.current_model_f1 = training_result['validation_metrics']['f1_macro']
        else:
            self.current_model_f1 = training_result['final_accuracy'] * 0.9  # Rough approximation
        
        # Record performance history
        self.model_performance_history.append({
            'timestamp': time.time(),
            'accuracy': self.current_model_accuracy,
            'f1_macro': self.current_model_f1,
            'loss': training_result['final_loss'],
            'epochs': epochs,
            'energy_consumed': training_state['energy_consumed']
        })
        
        return {
            'training_result': training_result,
            'energy_state': training_state,
            'model_size': self.fl_model.get_model_size(),
            'computational_complexity': self.fl_model.get_computational_complexity()
        }

    def get_model_weights(self) -> Dict[str, np.ndarray]:
        """Get current model weights for aggregation"""
        if self.fl_model is not None:
            return self.fl_model.get_model_weights()
        return {}

    def set_model_weights(self, weights: Dict[str, np.ndarray]):
        """Set model weights from global aggregation"""
        if self.fl_model is not None:
            self.fl_model.set_model_weights(weights)

    def evaluate_model(self, test_dataset) -> Dict[str, float]:
        """Evaluate current model performance"""
        if self.fl_model is not None:
            return self.fl_model.evaluate(test_dataset)
        return {}

    def update_dynamic_state(self, operation: str = "idle", data_size_mb: float = 0, duration_seconds: float = 1.0):
        """Update device state using dynamic resource modeling"""
        # Update resources using the resource manager
        resource_state = self.resource_manager.update_resources(operation, data_size_mb, duration_seconds)
        
        # Update the device state based on resource manager output
        self.state.current_battery = resource_state['battery_level']
        self.state.current_bandwidth_up = resource_state['bandwidth_up'] 
        self.state.current_bandwidth_down = resource_state['bandwidth_down']
        self.state.temperature = resource_state['cpu_performance']['temperature']
        
        # Update status based on battery and performance
        if not resource_state['battery_alive'] or resource_state['battery_level'] < 0.05:
            self.state.status = DeviceStatus.FAILED
        elif resource_state['cpu_performance']['thermal_factor'] < 0.5:
            self.state.status = DeviceStatus.SLEEPING  # Thermal protection
        else:
            self.state.status = DeviceStatus.ACTIVE
        
        # Update consumption history
        self.energy_consumption_history.append({
            'timestamp': time.time(),
            'operation': operation,
            'energy_consumed': resource_state['energy_consumed'],
            'battery_level_after': resource_state['battery_level']
        })

        if hasattr(self, 'carbon_tracker'):
            carbon_result = self.carbon_tracker.calculate_operation_carbon(
                operation, resource_state['energy_consumed'], data_size_mb, duration_seconds
            )
            
            # Add carbon info to consumption history
            self.carbon_footprint_history.append({
                'timestamp': time.time(),
                'operation': operation,
                'carbon_g': carbon_result['total_carbon_g'],
                'carbon_breakdown': carbon_result['carbon_breakdown']
            })
            
            # Add carbon result to the returned resource state
            resource_state['carbon_footprint'] = carbon_result
                    
        return resource_state

    def get_available_resources(self) -> Dict[str, float]:
        """Get current available resources for FL participation"""
        return {
            'battery_level': self.state.current_battery,
            'cpu_availability': 1.0 - self.state.current_cpu_load,
            'bandwidth_up': self.state.current_bandwidth_up,
            'bandwidth_down': self.state.current_bandwidth_down,
            'data_size': self.local_data_size,
            'data_quality': self.data_quality_score,
            'gradient_quality': self.gradient_quality_score
        }
    
    def is_available_for_training(self, min_battery=0.1, min_bandwidth=0.1) -> bool:
        """Check if device meets minimum requirements for FL participation"""
        return (self.state.status == DeviceStatus.ACTIVE and
                self.state.current_battery > min_battery and
                self.state.current_bandwidth_up > min_bandwidth and
                self.state.current_cpu_load < 0.9)
    
    def __str__(self):
        return f"Client_{self.client_id}(Battery: {self.state.current_battery:.2f}, Status: {self.state.status.value})"


#### 5. Client Selection model building

In [11]:
@dataclass
class SelectionCriteria:
    """Criteria weights for multi-objective optimization"""
    convergence_speed_weight: float = 0.25
    fairness_weight: float = 0.25
    energy_weight: float = 0.25
    network_lifetime_weight: float = 0.25
    
    def normalize(self):
        """Ensure weights sum to 1.0"""
        total = (self.convergence_speed_weight + self.fairness_weight + 
                self.energy_weight + self.network_lifetime_weight)
        if total > 0:
            self.convergence_speed_weight /= total
            self.fairness_weight /= total
            self.energy_weight /= total
            self.network_lifetime_weight /= total

In [12]:
class SelectionPolicy(ABC):
    """Abstract base class for client selection policies"""
    
    def __init__(self, name: str):
        self.name = name
        self.selection_history = []
        
    @abstractmethod
    def select_clients(self, available_clients: List, num_clients: int, **kwargs) -> List:
        """Select clients based on the policy"""
        pass
    
    def record_selection(self, selected_clients: List, round_num: int, criteria_scores: Dict = None):
        """Record selection for analysis"""
        self.selection_history.append({
            'round': round_num,
            'selected_clients': [client.client_id for client in selected_clients],
            'criteria_scores': criteria_scores or {},
            'timestamp': time.time()
        })


##### 5.1. Random

In [13]:
class RandomSelectionPolicy(SelectionPolicy):
    """Random client selection policy"""
    
    def __init__(self):
        super().__init__("Random")
    
    def select_clients(self, available_clients: List, num_clients: int, **kwargs) -> List:
        """Randomly select clients from available pool"""
        if len(available_clients) <= num_clients:
            return available_clients
        
        selected = random.sample(available_clients, num_clients)
        
        # Record selection with basic fairness score
        fairness_score = self._calculate_fairness_score(available_clients, selected)
        self.record_selection(selected, kwargs.get('round_num', 0), 
                            {'fairness': fairness_score})
        
        return selected
    
    def _calculate_fairness_score(self, available_clients: List, selected_clients: List) -> float:
        """Simple fairness measure: selection probability uniformity"""
        if not available_clients:
            return 1.0
        return len(selected_clients) / len(available_clients)


##### 5.2. Greedy

In [14]:
class GreedySelectionPolicy(SelectionPolicy):
    """Greedy client selection based on single best criterion"""

    def __init__(self, criterion: str = "data_quality"):
        super().__init__(f"Greedy_{criterion}")
        self.criterion = criterion

    def select_clients(
        self, available_clients: List[IoTClient], num_clients: int, **kwargs
    ) -> List:
        """Select clients greedily based on specified criterion"""
        if len(available_clients) <= num_clients:
            return available_clients

        # Score clients based on criterion
        scored_clients = []
        for client in available_clients:
            score = self._calculate_criterion_score(client, self.criterion)
            scored_clients.append((client, score))

        # Sort by score (descending) and select top clients
        scored_clients.sort(key=lambda x: x[1], reverse=True)
        selected = [client for client, score in scored_clients[:num_clients]]

        # Record selection metrics
        avg_score = np.mean([score for _, score in scored_clients[:num_clients]])
        fairness_score = self._calculate_fairness_score(available_clients, selected)

        self.record_selection(
            selected,
            kwargs.get("round_num", 0),
            {f"{self.criterion}_score": avg_score, "fairness": fairness_score},
        )

        return selected

    def _calculate_criterion_score(self, client: IoTClient, criterion: str) -> float:
        """Calculate score for specific criterion"""
        resources = client.get_available_resources()

        if criterion == "data_quality":
            return resources["data_quality"]
        elif criterion == "data_size":
            return resources["data_size"] / 1000.0  # Normalize
        elif criterion == "battery_level":
            return resources["battery_level"]
        elif criterion == "bandwidth":
            return resources["bandwidth_up"]
        elif criterion == "gradient_quality":
            return resources["gradient_quality"]
        elif criterion == "composite":
            # Simple composite score
            return (
                resources["data_quality"] * 0.3
                + resources["battery_level"] * 0.3
                + (resources["data_size"] / 1000.0) * 0.2
                + (resources["bandwidth_up"] / 100.0) * 0.2
            )

        return 0.0

    def _calculate_fairness_score(
        self, available_clients: List, selected_clients: List
    ) -> float:
        """Calculate fairness based on resource distribution"""
        if not available_clients or not selected_clients:
            return 0.0

        # Calculate coefficient of variation for battery levels
        selected_batteries = [
            client.state.current_battery for client in selected_clients
        ]
        available_batteries = [
            client.state.current_battery for client in available_clients
        ]

        selected_cv = (
            np.std(selected_batteries) / np.mean(selected_batteries)
            if np.mean(selected_batteries) > 0
            else 0
        )
        available_cv = (
            np.std(available_batteries) / np.mean(available_batteries)
            if np.mean(available_batteries) > 0
            else 0
        )

        # Lower CV is more fair (less variation)
        return max(0, 1.0 - abs(selected_cv - available_cv))

##### 5.3. Multi-Objective Optimization

In [15]:
class MOOSelectionPolicy(SelectionPolicy):
    """Multi-Objective Optimization client selection policy"""

    def __init__(
        self, criteria: SelectionCriteria = None, adaptive_weights: bool = True
    ):
        super().__init__("MOO")
        self.criteria = criteria or SelectionCriteria()
        self.criteria.normalize()
        self.adaptive_weights = adaptive_weights
        self.network_history = []

    def select_clients(
        self, available_clients: List, num_clients: int, **kwargs
    ) -> List:
        """Select clients using multi-objective optimization"""
        if len(available_clients) <= num_clients:
            return available_clients

        round_num = kwargs.get("round_num", 0)

        # Adapt weights if enabled
        if self.adaptive_weights:
            self._adapt_weights(available_clients, round_num)

        # Calculate multi-objective scores for all clients
        client_scores = []
        for client in available_clients:
            score = self._calculate_moo_score(client, available_clients)
            client_scores.append((client, score))

        # Select clients using Pareto-based selection or weighted sum
        selected = self._pareto_selection(client_scores, num_clients)

        # Calculate metrics for this selection
        metrics = self._calculate_selection_metrics(available_clients, selected)
        self.record_selection(selected, round_num, metrics)

        return selected

    def _adapt_weights(self, available_clients: List, round_num: int):
        """Dynamically adapt objective weights based on network state"""
        if not available_clients:
            return

        avg_battery = np.mean(
            [client.state.current_battery for client in available_clients]
        )
        min_battery = min(
            [client.state.current_battery for client in available_clients]
        )

        # Increase energy weight when average battery is low
        if avg_battery < 0.3:
            self.criteria.energy_weight = min(0.5, self.criteria.energy_weight * 1.5)
            self.criteria.network_lifetime_weight = min(
                0.3, self.criteria.network_lifetime_weight * 1.3
            )

        # Increase fairness weight if some devices have very low battery
        if min_battery < 0.1:
            self.criteria.fairness_weight = min(
                0.4, self.criteria.fairness_weight * 1.4
            )

        # Increase convergence weight in early rounds
        if round_num < 10:
            self.criteria.convergence_speed_weight = min(
                0.4, self.criteria.convergence_speed_weight * 1.2
            )

        # Normalize weights
        self.criteria.normalize()

    def _calculate_moo_score(
        self, client: IoTClient, all_clients: List
    ) -> Dict[str, float]:
        """Calculate multi-objective scores for a client"""
        resources = client.get_available_resources()

        # Objective 1: Convergence Speed (data quality + size + gradient quality)
        convergence_score = (
            resources["data_quality"] * 0.4
            + min(1.0, resources["data_size"] / 1000.0) * 0.3
            + resources["gradient_quality"] * 0.3
        )

        # Objective 2: Fairness (inverse of recent participation frequency)
        participation_freq = self._calculate_participation_frequency(client.client_id)
        fairness_score = max(0, 1.0 - participation_freq)

        # Objective 3: Energy Conservation (battery level + estimated lifetime)
        battery_score = resources["battery_level"]
        if hasattr(client, "resource_manager"):
            lifetime_hours = (
                client.resource_manager.battery_model.get_estimated_lifetime(50.0, 25.0)
            )
            lifetime_score = min(1.0, lifetime_hours / 24.0)  # Normalize to 24 hours
        else:
            lifetime_score = battery_score
        energy_score = battery_score * 0.6 + lifetime_score * 0.4

        # Objective 4: Network Lifetime (relative battery compared to network average)
        if all_clients:
            avg_battery = np.mean([c.state.current_battery for c in all_clients])
            network_lifetime_score = min(
                2.0, resources["battery_level"] / max(0.01, avg_battery)
            )
            network_lifetime_score = network_lifetime_score / 2.0  # Normalize to 0-1
        else:
            network_lifetime_score = battery_score

        # Weighted combination
        combined_score = (
            convergence_score * self.criteria.convergence_speed_weight
            + fairness_score * self.criteria.fairness_weight
            + energy_score * self.criteria.energy_weight
            + network_lifetime_score * self.criteria.network_lifetime_weight
        )

        return {
            "combined_score": combined_score,
            "convergence": convergence_score,
            "fairness": fairness_score,
            "energy": energy_score,
            "network_lifetime": network_lifetime_score,
        }

    def _calculate_participation_frequency(self, client_id: int) -> float:
        """Calculate how frequently a client has been selected recently"""
        if not self.selection_history:
            return 0.0

        recent_rounds = min(10, len(self.selection_history))  # Look at last 10 rounds
        recent_selections = self.selection_history[-recent_rounds:]

        participation_count = sum(
            1 for record in recent_selections if client_id in record["selected_clients"]
        )

        return participation_count / recent_rounds if recent_rounds > 0 else 0.0

    def _pareto_selection(self, client_scores: List[Tuple], num_clients: int) -> List:
        """Select clients using Pareto efficiency or weighted sum ranking"""
        # For simplicity, use weighted sum ranking
        # In a full implementation, you could use NSGA-II or similar for true Pareto selection

        # Sort by combined score
        client_scores.sort(key=lambda x: x[1]["combined_score"], reverse=True)

        # Select top clients, but ensure minimum diversity
        selected = []
        remaining_clients = [cs for cs in client_scores]

        # Select top 70% based on score
        top_count = max(1, int(num_clients * 0.7))
        selected.extend([client for client, score in remaining_clients[:top_count]])
        remaining_clients = remaining_clients[top_count:]

        # Select remaining 30% with diversity consideration
        while len(selected) < num_clients and remaining_clients:
            # Simple diversity: prefer clients with different characteristics
            best_client = remaining_clients[0][0]  # Fallback to best remaining

            # Remove selected client from remaining
            remaining_clients = [
                (c, s) for c, s in remaining_clients if c != best_client
            ]
            selected.append(best_client)

        return selected

    def _calculate_selection_metrics(
        self, available_clients: List, selected_clients: List
    ) -> Dict[str, float]:
        """Calculate comprehensive metrics for the selection"""
        if not selected_clients:
            return {}

        # Average scores for selected clients
        selected_scores = [
            self._calculate_moo_score(client, available_clients)
            for client in selected_clients
        ]

        metrics = {
            "avg_convergence_score": np.mean(
                [s["convergence"] for s in selected_scores]
            ),
            "avg_fairness_score": np.mean([s["fairness"] for s in selected_scores]),
            "avg_energy_score": np.mean([s["energy"] for s in selected_scores]),
            "avg_network_lifetime_score": np.mean(
                [s["network_lifetime"] for s in selected_scores]
            ),
            "avg_combined_score": np.mean(
                [s["combined_score"] for s in selected_scores]
            ),
        }

        # Additional fairness metrics
        selected_batteries = [c.state.current_battery for c in selected_clients]
        available_batteries = [c.state.current_battery for c in available_clients]

        metrics.update(
            {
                "battery_diversity": np.std(selected_batteries),
                "battery_fairness": 1.0
                - abs(np.mean(selected_batteries) - np.mean(available_batteries)),
                "selection_ratio": len(selected_clients) / len(available_clients),
                "current_weights": {
                    "convergence": self.criteria.convergence_speed_weight,
                    "fairness": self.criteria.fairness_weight,
                    "energy": self.criteria.energy_weight,
                    "network_lifetime": self.criteria.network_lifetime_weight,
                },
            }
        )

        return metrics

In [16]:
class SelectionPolicyManager:
    """Manager for different selection policies"""
    
    def __init__(self):
        self.policies = {}
        self.current_policy = None
        
    def add_policy(self, policy: SelectionPolicy):
        """Add a selection policy"""
        self.policies[policy.name] = policy
        
    def set_active_policy(self, policy_name: str):
        """Set the active selection policy"""
        if policy_name in self.policies:
            self.current_policy = self.policies[policy_name]
        else:
            raise ValueError(f"Policy {policy_name} not found")
    
    def select_clients(self, available_clients: List, num_clients: int, **kwargs) -> List:
        """Select clients using the active policy"""
        if self.current_policy is None:
            raise ValueError("No active policy set")
        
        return self.current_policy.select_clients(available_clients, num_clients, **kwargs)
    
    def get_policy_history(self, policy_name: str) -> List:
        """Get selection history for a specific policy"""
        if policy_name in self.policies:
            return self.policies[policy_name].selection_history
        return []
    
    def compare_policies(self, metrics: List[str] = None) -> Dict:
        """Compare performance across all policies"""
        if metrics is None:
            metrics = ['avg_combined_score', 'avg_fairness_score', 'avg_energy_score']
        
        comparison = {}
        for policy_name, policy in self.policies.items():
            if policy.selection_history:
                policy_metrics = {}
                for metric in metrics:
                    values = []
                    for record in policy.selection_history:
                        if metric in record.get('criteria_scores', {}):
                            values.append(record['criteria_scores'][metric])
                    
                    if values:
                        policy_metrics[metric] = {
                            'mean': np.mean(values),
                            'std': np.std(values),
                            'min': np.min(values),
                            'max': np.max(values)
                        }
                
                comparison[policy_name] = policy_metrics
        
        return comparison


#### 6. Gateway

In [17]:
class Gateway:
    """Gateway responsible for client selection"""
    
    def __init__(self, gateway_id: int, coverage_area: float = 1000.0):
        self.gateway_id = gateway_id
        self.coverage_area = coverage_area  # meters
        self.connected_clients: List[IoTClient] = []
        self.selection_history = []

        # Add policy manager
        self.policy_manager = SelectionPolicyManager()
        
        # Initialize with all policies
        self.policy_manager.add_policy(RandomSelectionPolicy())
        self.policy_manager.add_policy(GreedySelectionPolicy("composite"))
        self.policy_manager.add_policy(MOOSelectionPolicy())
        
        # Set default policy
        self.policy_manager.set_active_policy("Random")

        
    def add_client(self, client: IoTClient):
        """Add a client to this gateway's coverage"""
        if client not in self.connected_clients:
            self.connected_clients.append(client)
    
    def remove_client(self, client: IoTClient):
        """Remove a client from this gateway's coverage"""
        if client in self.connected_clients:
            self.connected_clients.remove(client)
    
    def get_available_clients(self) -> List[IoTClient]:
        """Get all clients that are available for training"""
        return [client for client in self.connected_clients 
                if client.is_available_for_training()]
    
    def select_clients(self, selection_policy: str, num_clients: int, **kwargs) -> List[IoTClient]:
        """Select clients based on the specified policy"""
        available_clients = self.get_available_clients()
        
        if len(available_clients) <= num_clients:
            return available_clients
        
        # Set the active policy
        if selection_policy in self.policy_manager.policies:
            self.policy_manager.set_active_policy(selection_policy)
        else:
            print(f"Warning: Policy {selection_policy} not found, using Random")
            self.policy_manager.set_active_policy("Random")
        
        # Use policy manager to select clients
        selected_clients = self.policy_manager.select_clients(available_clients, num_clients, **kwargs)
        
        # Record in gateway's own history
        self.selection_history.append({
            'round': kwargs.get('round_num', 0),
            'policy': selection_policy,
            'selected': [c.client_id for c in selected_clients],
            'available': len(available_clients),
            'timestamp': time.time()
        })
        
        return selected_clients



#### 5. Central/Cloud Server

In [18]:
class CentralServer:
    """Central server responsible for model aggregation"""
    
    def __init__(self):
        self.global_model = None
        self.training_round = 0
        self.training_history = []
        self.connected_gateways: List[Gateway] = []
        
    def add_gateway(self, gateway: Gateway):
        """Add a gateway to the network"""
        if gateway not in self.connected_gateways:
            self.connected_gateways.append(gateway)
    
    def get_all_clients(self) -> List[IoTClient]:
        """Get all clients across all gateways"""
        all_clients = []
        for gateway in self.connected_gateways:
            all_clients.extend(gateway.connected_clients)
        return all_clients
    
    def aggregate_models(self, client_updates: List[Tuple[IoTClient, Dict]]) -> Dict:
        """Perform federated averaging - simplified version"""
        if not client_updates:
            return {}
        
        # Simple weighted averaging based on local data size
        total_data_size = sum(client.local_data_size for client, _ in client_updates)
        
        # Placeholder aggregation logic
        aggregated_update = {
            'accuracy': np.mean([update['accuracy'] for _, update in client_updates]),
            'loss': np.mean([update['loss'] for _, update in client_updates]),
            'participants': len(client_updates),
            'total_data_size': total_data_size
        }
        
        return aggregated_update


#### 6. Simulator

In [19]:
class FLSystem:
    """Main federated learning system orchestrator"""
    
    def __init__(self):
        self.server = CentralServer()
        self.gateways: List[Gateway] = []
        self.clients: List[IoTClient] = []
        self.system_metrics = {
            'round_metrics': [],
            'energy_consumption': [],
            'carbon_footprint': [],
            'network_lifetime': 0,
            'failed_devices': []
        }

        self.system_carbon = SystemCarbonFootprint("global")
    
        # Add carbon metrics to system_metrics
        self.system_metrics.update({
            'total_carbon_emissions_g': 0.0,
            'carbon_per_round': [],
            'carbon_efficiency': []
        })
        self.aggregator = FederatedAggregator("fedavg")
        self.global_model_history = []
        self.test_dataset = None
        self.class_names = []
        
        # FL training configuration
        self.fl_config = {
            'model_type': 'cnn',
            'input_size': 25,
            'num_classes': 10,
            'epochs_per_round': 3,
            'batch_size': 32
        }
        
        # Performance tracking
        self.system_metrics.update({
            'global_accuracy_history': [],
            'global_f1_history': [],
            'global_loss_history': [],
            'round_training_times': [],
            'convergence_metrics': []
        })

    def initialize_fl_system(self, X_data: np.ndarray, y_data: np.ndarray, class_names: List[str],
                        test_size: float = 0.2, non_iid_factor: float = 0.7):
        """Initialize the federated learning system with data"""
        
        # Store configuration
        self.fl_config['input_size'] = X_data.shape[1]
        self.fl_config['num_classes'] = len(class_names)
        self.class_names = class_names
        
        # Split data into train/test
        # X_train, X_test, y_train, y_test = train_test_split(
        #     X_data, y_data, test_size=test_size, random_state=42, stratify=y
        # )
        
        # Load the local MNIST dataset as placeholder        

        (X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
        X_train = X_train.reshape(-1, 28*28) / 255.0
        X_test = X_test.reshape(-1, 28*28) / 255.0
        y_train = y_train.astype(np.int32)
        y_test = y_test.astype(np.int32)
        
        # Number of classes of the MNIST dataset
        num_classes = 10

        # Create global test dataset
        dummy_model = FLModel("cnn", X_data.shape[1], len(class_names), "mobile")
        self.test_dataset, _ = dummy_model.prepare_data(X_test, y_test, fit_scaler=True)
        
        # Distribute data among clients (Non-IID)
        self._distribute_data_non_iid(X_train, y_train, non_iid_factor)
        
        # Initialize FL models for all clients
        for client in self.clients:
            client.initialize_fl_model(
                self.fl_config['model_type'],
                self.fl_config['input_size'],
                self.fl_config['num_classes']
            )
        self._update_client_data_quality()
    
        print(f"FL System initialized: {len(self.clients)} clients, {len(class_names)} classes")
        print(f"Test dataset: {len(self.test_dataset)} samples")

    def _distribute_data_non_iid(self, X_train: np.ndarray, y_train: np.ndarray, non_iid_factor: float):
        """Distribute training data among clients using Dirichlet distribution"""
        
        n_clients = len(self.clients)
        
        if non_iid_factor > 0:
            # Use Dirichlet distribution for non-IID split
            alpha = (1.0 - non_iid_factor) * 10  # Convert factor to alpha parameter
            alpha = max(0.1, alpha)  # Ensure minimum alpha
            
            print(f"Creating Dirichlet split with alpha={alpha:.2f}")
            client_indices = create_dirichlet_split(y_train, n_clients, alpha)
            
        else:
            # IID distribution
            print("Creating IID split")
            indices = np.random.permutation(len(y_train))
            client_indices = np.array_split(indices, n_clients)
        
        # Assign data to clients
        for i, client in enumerate(self.clients):
            if len(client_indices[i]) > 0:
                client_X = X_train[client_indices[i]]
                client_y = y_train[client_indices[i]]
                client.set_local_data(client_X, client_y)
                
                # Print client data distribution
                unique, counts = np.unique(client_y, return_counts=True)
                print(f"Client {client.client_id} ({client.specs.device_type}): {len(client_y)} samples")
                
                # Show class distribution
                class_dist = {}
                for c, count in zip(unique, counts):
                    if c < len(self.class_names):
                        class_dist[self.class_names[c]] = count
                
                # Show top 3 classes
                top_classes = sorted(class_dist.items(), key=lambda x: x[1], reverse=True)[:3]
                print(f"  Top classes: {dict(top_classes)}")
                
                # Calculate and print non-IID metric (entropy)
                proportions = counts / len(client_y)
                entropy = -np.sum(proportions * np.log(proportions + 1e-12))
                max_entropy = np.log(len(unique))
                normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0
                print(f"  Data diversity: {normalized_entropy:.3f} (1.0 = perfectly balanced)")
    
    def _update_client_data_quality(self):
        """Update client data quality scores after FL system initialization"""
        for client in self.clients:
            if client.local_dataset is not None and hasattr(client, 'local_data_size'):
                # Recalculate with correct number of classes
                if hasattr(client, '_local_y'):  # We'll store this in set_local_data
                    unique_classes = len(np.unique(client._local_y))
                else:
                    unique_classes = 1  # Fallback
                    
                data_size_factor = min(1.0, client.local_data_size / 1000.0)
                class_diversity_factor = unique_classes / max(1, self.fl_config['num_classes'])
                
                client.data_quality_score = min(1.0, (data_size_factor * 0.6 + class_diversity_factor * 0.4))
                
                # Update gradient quality
                device_capability_factors = {
                    "sensor": 0.7,
                    "mobile": 0.85,
                    "gateway": 1.0
                }
                base_factor = device_capability_factors.get(client.specs.device_type, 0.8)
                client.gradient_quality_score = client.data_quality_score * base_factor

    def run_federated_round(self, selection_policy: str, num_clients: int, round_num: int) -> Dict[str, any]:
        """Run a complete federated learning round"""
        
        # Client selection
        gateway = self.gateways[0]
        selected_clients = gateway.select_clients(selection_policy, num_clients, round_num=round_num)
        
        if not selected_clients:
            return {'error': 'No clients available'}
        
        print(f"\nFederated Round {round_num} ({selection_policy})")
        print(f"Selected clients: {[c.client_id for c in selected_clients]}")
        
        # Local training phase
        client_models = []
        round_metrics = {
            'round_num': round_num,
            'policy': selection_policy,
            'selected_clients': [c.client_id for c in selected_clients],
            'client_results': [],
            'total_energy': 0.0,
            'total_carbon': 0.0,
            'training_time': 0.0
        }
        
        start_time = time.time()
        
        for client in selected_clients:
            print(f"  Training client {client.client_id}...")
            
            # Local training
            training_result = client.train_local_model(
                epochs=self.fl_config['epochs_per_round'],
                batch_size=self.fl_config['batch_size']
            )
            
            if 'error' in training_result:
                print(f"    Error: {training_result['error']}")
                continue
            
            # Collect model for aggregation
            model_weights = client.get_model_weights()
            client_metadata = {
                'data_size': client.local_data_size,
                'training_accuracy': training_result['training_result']['final_accuracy'],
                'device_type': client.specs.device_type,
                'energy_consumed': training_result['energy_state']['energy_consumed'],
                'data_quality': client.data_quality_score,
                'gradient_quality': client.gradient_quality_score
            }
            
            client_models.append((client.fl_model, client_metadata))
            
            # Track metrics
            round_metrics['total_energy'] += client_metadata['energy_consumed']
            if 'carbon_footprint' in training_result['energy_state']:
                round_metrics['total_carbon'] += training_result['energy_state']['carbon_footprint']['total_carbon_g']
            
            round_metrics['client_results'].append({
                'client_id': client.client_id,
                'accuracy': client_metadata['training_accuracy'],
                'energy': client_metadata['energy_consumed'],
                'data_size': client_metadata['data_size']
            })
        
        training_time = time.time() - start_time
        round_metrics['training_time'] = training_time
        
        if not client_models:
            return {'error': 'No successful client training'}
        
        # Aggregation phase
        print(f"  Aggregating {len(client_models)} models...")
        aggregated_weights = self.aggregator.aggregate_models(client_models)
        
        # Update global model (use first client's model as template)
        template_client = selected_clients[0]
        template_client.set_model_weights(aggregated_weights)
        
        # Evaluate global model
        global_metrics = template_client.evaluate_model(self.test_dataset)
        round_metrics['global_metrics'] = global_metrics
        
        # Update system metrics
        self.system_metrics['global_accuracy_history'].append(global_metrics['accuracy'])
        self.system_metrics['global_f1_history'].append(global_metrics['f1_macro'])
        self.system_metrics['global_loss_history'].append(global_metrics['loss'])
        self.system_metrics['round_training_times'].append(training_time)
        
        # Record global model state
        self.global_model_history.append({
            'round': round_num,
            'accuracy': global_metrics['accuracy'],
            'f1_macro': global_metrics['f1_macro'],
            'f1_micro': global_metrics['f1_micro'],
            'precision_macro': global_metrics['precision_macro'],
            'recall_macro': global_metrics['recall_macro'],
            'loss': global_metrics['loss'],
            'participating_clients': len(client_models),
            'total_energy': round_metrics['total_energy'],
            'total_carbon': round_metrics['total_carbon']
        })
        
        print(f"  Global Accuracy: {global_metrics['accuracy']:.4f}")
        print(f"  Global F1-macro: {global_metrics['f1_macro']:.4f}")
        print(f"  Round Energy: {round_metrics['total_energy']:.1f} mAh")
        
        return round_metrics

    def run_fl_experiment(self, policies: List[str], num_rounds: int = 15, clients_per_round: int = 5) -> Dict[str, any]:
        """Run complete FL experiment comparing different policies"""
        
        results = {}
        
        for policy in policies:
            print(f"\n{'='*50}")
            print(f"Running FL Experiment: {policy} Policy")
            print(f"{'='*50}")
            
            # Reset system state
            for client in self.clients:
                client.state.current_battery = 0.9 + 0.1 * np.random.random()
                client.state.status = DeviceStatus.ACTIVE
                if hasattr(client, 'carbon_tracker'):
                    client.carbon_tracker.total_carbon_emissions_g = 0.0
            
            policy_results = {
                'rounds': [],
                'accuracy_history': [],
                'f1_history': [],
                'loss_history': [],
                'energy_consumption': [],
                'carbon_emissions': [],
                'network_lifetime': [],
                'convergence_round': None,
                'final_metrics': {}
            }
            
            convergence_threshold = 0.02  # Convergence if accuracy improvement < 2%
            convergence_patience = 3
            no_improvement_count = 0
            best_accuracy = 0.0
            
            for round_num in range(num_rounds):
                # Run federated round
                round_result = self.run_federated_round(policy, clients_per_round, round_num)
                
                if 'error' in round_result:
                    print(f"Round {round_num} failed: {round_result['error']}")
                    break
                
                # Extract metrics
                global_metrics = round_result['global_metrics']
                current_accuracy = global_metrics['accuracy']
                
                policy_results['rounds'].append(round_num)
                policy_results['accuracy_history'].append(current_accuracy)
                policy_results['f1_history'].append(global_metrics['f1_macro'])
                policy_results['loss_history'].append(global_metrics['loss'])
                policy_results['energy_consumption'].append(round_result['total_energy'])
                policy_results['carbon_emissions'].append(round_result['total_carbon'])
                
                # Network lifetime
                active_clients = sum(1 for c in self.clients if c.state.status == DeviceStatus.ACTIVE)
                policy_results['network_lifetime'].append(active_clients / len(self.clients))
                
                # Check convergence
                if current_accuracy > best_accuracy + convergence_threshold:
                    best_accuracy = current_accuracy
                    no_improvement_count = 0
                else:
                    no_improvement_count += 1
                
                if no_improvement_count >= convergence_patience and policy_results['convergence_round'] is None:
                    policy_results['convergence_round'] = round_num
                    print(f"  Convergence detected at round {round_num}")
                
                # Early stopping if network fails
                if active_clients < len(self.clients) * 0.3:  # Less than 30% clients active
                    print(f"  Network failure at round {round_num} ({active_clients} clients active)")
                    break
            
            # Final metrics
            if policy_results['accuracy_history']:
                policy_results['final_metrics'] = {
                    'final_accuracy': policy_results['accuracy_history'][-1],
                    'final_f1_macro': policy_results['f1_history'][-1],
                    'final_loss': policy_results['loss_history'][-1],
                    'total_energy': sum(policy_results['energy_consumption']),
                    'total_carbon': sum(policy_results['carbon_emissions']),
                    'final_network_lifetime': policy_results['network_lifetime'][-1],
                    'rounds_completed': len(policy_results['rounds']),
                    'convergence_efficiency': policy_results['convergence_round'] or len(policy_results['rounds'])
                }
            
            results[policy] = policy_results
            
            # Print policy summary
            if policy_results['final_metrics']:
                fm = policy_results['final_metrics']
                print(f"\n{policy} Policy Summary:")
                print(f"  Final Accuracy: {fm['final_accuracy']:.4f}")
                print(f"  Final F1-macro: {fm['final_f1_macro']:.4f}")
                print(f"  Total Energy: {fm['total_energy']:.1f} mAh")
                print(f"  Total Carbon: {fm['total_carbon']:.2f}g CO2eq")
                print(f"  Network Survival: {fm['final_network_lifetime']*100:.1f}%")
                print(f"  Convergence: Round {fm['convergence_efficiency']}")
        
        return results
        
    def add_gateway(self, gateway: Gateway):
        """Add gateway to the system"""
        self.gateways.append(gateway)
        self.server.add_gateway(gateway)
    
    def add_client(self, client: IoTClient, gateway_id: int = 0):
        """Add client to the system and assign to a gateway"""
        self.clients.append(client)
        if gateway_id < len(self.gateways):
            self.gateways[gateway_id].add_client(client)
    
    def get_system_status(self) -> Dict:
        """Get overall system status"""
        total_clients = len(self.clients)
        active_clients = sum(1 for client in self.clients if client.state.status == DeviceStatus.ACTIVE)
        avg_battery = np.mean([client.state.current_battery for client in self.clients])
        
        return {
            'total_clients': total_clients,
            'active_clients': active_clients,
            'failed_clients': total_clients - active_clients,
            'average_battery_level': avg_battery,
            'training_round': self.server.training_round,
            'network_alive': active_clients > 0
        }
    
    def simulate_time_step(self, time_step_seconds: float = 60.0):
        """Simulate one time step for all devices in the system"""
        failed_devices = []
        
        for client in self.clients:
            # Most devices are idle most of the time
            operation = "idle"
            
            # Simulate random environmental changes
            if np.random.random() < 0.1:  # 10% chance of environmental change
                ambient_temp = 20 + 15 * np.random.random()  # 20-35°C
                interference = 0.05 + 0.3 * np.random.random()  # 5-35% interference
                client.resource_manager.set_environmental_conditions(ambient_temp, interference)
            
            # Update device state
            resource_state = client.update_dynamic_state(operation, 0, time_step_seconds)
            
            # Track failed devices
            if client.state.status == DeviceStatus.FAILED:
                failed_devices.append(client.client_id)
        
        # Update system metrics
        if failed_devices:
            self.system_metrics['failed_devices'].extend(failed_devices)
            print(f"⚠️  Devices failed in this time step: {failed_devices}")
        
        return len(failed_devices)
    
    def simulate_fl_round(self, selection_policy: str, num_clients: int, round_num: int) -> Dict[str, any]:
        """Simulate a complete FL training round with carbon tracking"""
        
        # Client selection
        gateway = self.gateways[0]
        selected_clients = gateway.select_clients(selection_policy, num_clients, round_num=round_num)
        
        if not selected_clients:
            return {'error': 'No clients available'}
        
        # Track participating clients for carbon calculation
        participating_ids = [client.client_id for client in selected_clients]
        
        round_metrics = {
            'round_num': round_num,
            'policy': selection_policy,
            'selected_clients': participating_ids,
            'energy_consumption': 0.0,
            'carbon_emissions': 0.0,
            'client_details': []
        }
        
        # Simulate FL operations for each selected client
        for client in selected_clients:
            client_metrics = {}
            
            # 1. Model download (server to client)
            download_state = client.update_dynamic_state("model_download", data_size_mb=5.0, duration_seconds=10.0)
            client_metrics['download'] = {
                'energy': download_state['energy_consumed'],
                'carbon': download_state.get('carbon_footprint', {}).get('total_carbon_g', 0)
            }
            
            # 2. Local training
            training_state = client.update_dynamic_state("training", data_size_mb=0, duration_seconds=90.0)
            client_metrics['training'] = {
                'energy': training_state['energy_consumed'],
                'carbon': training_state.get('carbon_footprint', {}).get('total_carbon_g', 0)
            }
            
            # 3. Model upload (client to server)
            upload_state = client.update_dynamic_state("communication", data_size_mb=2.0, duration_seconds=8.0)
            client_metrics['upload'] = {
                'energy': upload_state['energy_consumed'],
                'carbon': upload_state.get('carbon_footprint', {}).get('total_carbon_g', 0)
            }
            
            # Calculate client totals
            client_total_energy = (client_metrics['download']['energy'] + 
                                client_metrics['training']['energy'] + 
                                client_metrics['upload']['energy'])
            
            client_total_carbon = (client_metrics['download']['carbon'] + 
                                client_metrics['training']['carbon'] + 
                                client_metrics['upload']['carbon'])
            
            client_metrics['totals'] = {
                'energy': client_total_energy,
                'carbon': client_total_carbon,
                'battery_after': client.state.current_battery,
                'status': client.state.status.value
            }
            
            round_metrics['energy_consumption'] += client_total_energy
            round_metrics['carbon_emissions'] += client_total_carbon
            round_metrics['client_details'].append({
                'client_id': client.client_id,
                'metrics': client_metrics
            })
        
        # Record round in system carbon tracker
        carbon_round_metrics = self.system_carbon.record_training_round(
            round_num, participating_ids, server_energy_kwh=0.08  # Server aggregation energy
        )
        
        # Update system metrics
        self.system_metrics['total_carbon_emissions_g'] += carbon_round_metrics['round_carbon_g']
        self.system_metrics['carbon_per_round'].append(carbon_round_metrics['round_carbon_g'])
        self.system_metrics['carbon_efficiency'].append(carbon_round_metrics['carbon_efficiency'])
        
        # Add carbon metrics to round results
        round_metrics.update({
            'server_carbon': carbon_round_metrics['server_carbon_g'],
            'total_round_carbon': carbon_round_metrics['round_carbon_g'],
            'carbon_per_client': carbon_round_metrics['carbon_per_client']
        })
        
        return round_metrics

    def get_carbon_analysis(self) -> Dict[str, any]:
        """Get comprehensive carbon footprint analysis"""
        system_summary = self.system_carbon.get_system_carbon_summary()
        
        # Add client-specific carbon details
        client_carbon_details = {}
        for client in self.clients:
            if hasattr(client, 'carbon_tracker'):
                client_summary = client.carbon_tracker.get_carbon_summary()
                client_carbon_details[client.client_id] = {
                    'device_type': client.specs.device_type,
                    'total_carbon_g': client_summary['total_carbon_emissions_g'],
                    'carbon_breakdown': client_summary['emissions_by_operation'],
                    'energy_efficiency': client_summary['carbon_intensity_avg'],
                    'participation_count': len(client.participation_history)
                }
        
        return {
            'system_summary': system_summary,
            'client_details': client_carbon_details,
            'regional_comparisons': self.system_carbon.compare_carbon_scenarios(
                ["nordic", "europe", "usa", "coal_heavy"]
            )
        }
    
    def __str__(self):
        status = self.get_system_status()
        return (f"FL System: {status['active_clients']}/{status['total_clients']} clients active, "
                f"Round {status['training_round']}, Avg Battery: {status['average_battery_level']:.2f}")
    
    def run_policy_comparison(self, policies: List[str], num_rounds: int = 10, clients_per_round: int = 5):
        """Run comparison between different selection policies with carbon tracking"""
        results = {}
        
        for policy in policies:
            print(f"\n=== Testing {policy} Policy ===")
            policy_results = {
                'rounds': [],
                'energy_consumption': [],
                'carbon_emissions': [],  # Add carbon tracking
                'fairness_scores': [],
                'network_lifetime': []
            }
            
            # Reset system state for fair comparison
            for client in self.clients:
                client.state.current_battery = 0.8 + 0.2 * np.random.random()
                client.state.status = DeviceStatus.ACTIVE
                # Reset carbon tracker
                if hasattr(client, 'carbon_tracker'):
                    client.carbon_tracker.total_carbon_emissions_g = 0.0
                    client.carbon_tracker.emissions_history = []
            
            for round_num in range(num_rounds):
                # Use the new simulate_fl_round method
                round_result = self.simulate_fl_round(policy, clients_per_round, round_num)
                
                if 'error' in round_result:
                    print(f"Round {round_num}: {round_result['error']}")
                    break
                
                # Record metrics
                policy_results['rounds'].append(round_num)
                policy_results['energy_consumption'].append(round_result['energy_consumption'])
                policy_results['carbon_emissions'].append(round_result['total_round_carbon'])
                
                # Calculate system metrics
                avg_battery = np.mean([c.state.current_battery for c in self.clients])
                active_clients = sum(1 for c in self.clients if c.state.status == DeviceStatus.ACTIVE)
                policy_results['network_lifetime'].append(active_clients / len(self.clients))
                
                # Get fairness score from policy
                if self.gateways[0].policy_manager.current_policy.selection_history:
                    last_record = self.gateways[0].policy_manager.current_policy.selection_history[-1]
                    fairness = last_record.get('criteria_scores', {}).get('avg_fairness_score', 0)
                    policy_results['fairness_scores'].append(fairness)
                
                print(f"Round {round_num}: Energy={round_result['energy_consumption']:.1f}mAh, "
                    f"Carbon={round_result['total_round_carbon']:.2f}g CO2eq, "
                    f"Active={active_clients}/{len(self.clients)}")
            
            results[policy] = policy_results
        
        return results


#### 7. Run simulation

In [20]:
# Example usage and testing

# Create a simple FL system
fl_system = FLSystem()

# Add a gateway
gateway = Gateway(gateway_id=0)
fl_system.add_gateway(gateway)

# Create some IoT clients with different specs
device_specs = [
    DeviceSpecs(3000, 1.0, 10, 50, 1.0, "sensor"),      # Low-end sensor
    DeviceSpecs(5000, 1.5, 20, 100, 2.0, "gateway"),    # Mid-range gateway
    DeviceSpecs(4000, 1.2, 15, 80, 1.5, "mobile"),      # Mobile device
]

# Add clients to the system
for i, specs in enumerate(device_specs):
    for j in range(3):  # 3 devices of each type
        client_id = i * 3 + j
        client = IoTClient(client_id, specs)
        client.local_data_size = random.randint(100, 1000)  # Random data size
        client.data_quality_score = random.uniform(0.7, 1.0)  # Random quality
        fl_system.add_client(client, gateway_id=0)

# Display system status
print("=== Federated Learning System Initialized ===")
print(fl_system)
print(f"\nClients in Gateway 0: {len(gateway.connected_clients)}")

# Show some client details
print("\n=== Sample Client Details ===")
for i, client in enumerate(fl_system.clients[:3]):
    print(f"{client}")
    resources = client.get_available_resources()
    print(f"  Resources: Battery={resources['battery_level']:.2f}, "
            f"Data Size={resources['data_size']}, Quality={resources['data_quality']:.2f}")
# =========================================================
# Test client selection
available_clients = gateway.get_available_clients()
print(f"\nAvailable clients for training: {len(available_clients)}")

selected = gateway.select_clients("random", 3)
print(f"Selected clients: {[client.client_id for client in selected]}")

# Test dynamic resource modeling integration
print("\n=== Testing Dynamic Resource Modeling Integration ===")

# Simulate some time steps
print(f"Initial system status: {fl_system.get_system_status()}")

for step in range(5):
    print(f"\n--- Time Step {step + 1} ---")
    failed_count = fl_system.simulate_time_step(30.0)  # 30-second time steps
    
    status = fl_system.get_system_status()
    print(f"Active: {status['active_clients']}, Failed: {status['failed_clients']}, "
          f"Avg Battery: {status['average_battery_level']:.3f}")
    
    # Show detailed info for first client
    client = fl_system.clients[0]
    print(f"Client 0: Battery={client.state.current_battery:.3f}, "
          f"Temp={client.state.temperature:.1f}°C, Status={client.state.status.value}")

# Test FL training operation impact
print(f"\n--- Simulating FL Training ---")
selected_client = fl_system.clients[0]
print(f"Before training: Battery={selected_client.state.current_battery:.3f}")

# Simulate training operation (high energy consumption)
selected_client.update_dynamic_state("training", 0, 60)  # 1 minute of training
print(f"After training: Battery={selected_client.state.current_battery:.3f}")

# Simulate model download
selected_client.update_dynamic_state("model_download", 10, 5)  # Download 10MB model
print(f"After model download: Battery={selected_client.state.current_battery:.3f}")

# ========================================================================
# Test client selection policies
print("=== Testing Client Selection Policies ===")

# Show current system state
status = fl_system.get_system_status()
print(f"System status: {status['active_clients']}/{status['total_clients']} active, "
      f"Avg battery: {status['average_battery_level']:.3f}")

# Test individual policy selection
gateway = fl_system.gateways[0]
available = gateway.get_available_clients()
print(f"\nAvailable clients: {len(available)}")

# Test different policies
policies_to_test = ["Random", "Greedy_composite", "MOO"]

for policy in policies_to_test:
    print(f"\n--- {policy} Policy ---")
    selected = gateway.select_clients(policy, 3, round_num=1)
    
    print(f"Selected: {[c.client_id for c in selected]}")
    for client in selected:
        resources = client.get_available_resources()
        print(f"  Client {client.client_id}: Battery={resources['battery_level']:.2f}, "
              f"Quality={resources['data_quality']:.2f}, Data={resources['data_size']}")

# Run policy comparison
print(f"\n=== Running Policy Comparison ===")
comparison_results = fl_system.run_policy_comparison(policies_to_test, num_rounds=5, clients_per_round=3)

# Display comparison results
for policy, results in comparison_results.items():
    print(f"\n{policy} Policy Results:")
    if results['energy_consumption']:
        print(f"  Avg Energy per Round: {np.mean(results['energy_consumption']):.1f} mAh")
        print(f"  Final Network Lifetime: {results['network_lifetime'][-1]*100:.1f}%")
        if results['fairness_scores']:
            print(f"  Avg Fairness Score: {np.mean(results['fairness_scores']):.3f}")

# =====================================================
# Test Carbon Footprint Integration
print("=== Testing Carbon Footprint Integration ===")

# Initialize carbon trackers for existing clients
for client in fl_system.clients:
    fl_system.system_carbon.add_client_tracker(client.client_id, client.specs.device_type)

print(f"Initialized carbon tracking for {len(fl_system.clients)} clients")

# Run simulated FL rounds with different policies
policies_to_test = ["Random", "Greedy_composite", "MOO"]
results = {}

for policy in policies_to_test:
    print(f"\n=== Testing {policy} Policy with Carbon Tracking ===")
    policy_results = []
    
    # Reset system state for fair comparison
    for client in fl_system.clients:
        client.state.current_battery = 0.9 + 0.1 * np.random.random()
        client.state.status = DeviceStatus.ACTIVE
    
    # Run 3 rounds
    for round_num in range(3):
        round_result = fl_system.simulate_fl_round(policy, 3, round_num)
        
        if 'error' not in round_result:
            policy_results.append(round_result)
            
            print(f"Round {round_num}:")
            print(f"  Selected: {round_result['selected_clients']}")
            print(f"  Energy: {round_result['energy_consumption']:.1f} mAh")
            print(f"  Carbon: {round_result['total_round_carbon']:.2f}g CO2eq")
            print(f"  Carbon/client: {round_result['carbon_per_client']:.2f}g CO2eq")
            
            # Show client details
            for detail in round_result['client_details']:
                client_id = detail['client_id']
                totals = detail['metrics']['totals']
                print(f"    Client {client_id}: {totals['carbon']:.2f}g CO2eq, "
                      f"Battery: {totals['battery_after']:.3f}")
    
    results[policy] = policy_results

# Comprehensive carbon analysis
print(f"\n=== Carbon Footprint Analysis ===")
carbon_analysis = fl_system.get_carbon_analysis()

system_summary = carbon_analysis['system_summary']
print(f"Total system carbon: {system_summary['system_totals']['total_carbon_g']:.2f}g CO2eq")
print(f"Average carbon per round: {system_summary['carbon_efficiency_metrics']['carbon_per_round_g']:.2f}g CO2eq")
print(f"Carbon intensity: {system_summary['carbon_efficiency_metrics']['carbon_intensity_gco2_per_kwh']:.1f}g CO2eq/kWh")

print(f"\nOperation breakdown:")
for op, percentage in system_summary['operation_breakdown_percentage'].items():
    print(f"  {op.capitalize()}: {percentage:.1f}%")

# Compare policies by carbon efficiency
print(f"\n=== Policy Carbon Comparison ===")
policy_carbon_summary = {}
for policy, policy_results in results.items():
    if policy_results:
        total_carbon = sum(r['total_round_carbon'] for r in policy_results)
        avg_carbon_per_round = total_carbon / len(policy_results)
        avg_energy = np.mean([r['energy_consumption'] for r in policy_results])
        carbon_efficiency = total_carbon / max(avg_energy, 0.001)  # gCO2eq per mAh
        
        policy_carbon_summary[policy] = {
            'total_carbon': total_carbon,
            'avg_per_round': avg_carbon_per_round,
            'avg_energy': avg_energy,
            'efficiency': carbon_efficiency
        }
        
        print(f"{policy}:")
        print(f"  Total carbon: {total_carbon:.2f}g CO2eq")
        print(f"  Avg per round: {avg_carbon_per_round:.2f}g CO2eq")
        print(f"  Carbon efficiency: {carbon_efficiency:.3f}g CO2eq/mAh")

# Regional impact comparison
print(f"\n=== Regional Carbon Impact ===")
for region, data in carbon_analysis['regional_comparisons'].items():
    if data['percentage_reduction'] != 0:
        print(f"{region.capitalize()}: {data['percentage_reduction']:+.1f}% "
              f"({data['total_carbon_reduction_g']:+.1f}g CO2eq)")

# Client-level carbon analysis
print(f"\n=== Top Carbon Emitters ===")
client_carbon_sorted = sorted(
    carbon_analysis['client_details'].items(),
    key=lambda x: x[1]['total_carbon_g'],
    reverse=True
)

for client_id, details in client_carbon_sorted[:3]:
    print(f"Client {client_id} ({details['device_type']}):")
    print(f"  Total: {details['total_carbon_g']:.2f}g CO2eq")
    print(f"  Efficiency: {details['energy_efficiency']:.1f}g CO2eq/kWh")
    breakdown = details['carbon_breakdown']
    top_emission = max(breakdown, key=breakdown.get)
    print(f"  Top source: {top_emission} ({breakdown[top_emission]:.1f}g CO2eq)")

=== Federated Learning System Initialized ===
FL System: 9/9 clients active, Round 0, Avg Battery: 1.00

Clients in Gateway 0: 9

=== Sample Client Details ===
Client_0(Battery: 1.00, Status: active)
  Resources: Battery=1.00, Data Size=874, Quality=0.72
Client_1(Battery: 1.00, Status: active)
  Resources: Battery=1.00, Data Size=842, Quality=0.95
Client_2(Battery: 1.00, Status: active)
  Resources: Battery=1.00, Data Size=634, Quality=0.72

Available clients for training: 9
Selected clients: [8, 4, 5]

=== Testing Dynamic Resource Modeling Integration ===
Initial system status: {'total_clients': 9, 'active_clients': 9, 'failed_clients': 0, 'average_battery_level': 1.0, 'training_round': 0, 'network_alive': True}

--- Time Step 1 ---
Active: 9, Failed: 0, Avg Battery: 1.000
Client 0: Battery=1.000, Temp=35.6°C, Status=active

--- Time Step 2 ---
Active: 9, Failed: 0, Avg Battery: 1.000
Client 0: Battery=1.000, Temp=35.9°C, Status=active

--- Time Step 3 ---
Active: 9, Failed: 0, Avg Ba

In [21]:
# Test FL Model and Aggregation Integration
print("=== Testing FL Model and Aggregation Integration ===")

# Load real dataset instead of synthetic
print("Loading Fashion-MNIST dataset...")
X, y, class_names = load_fashion_mnist_data()

# Alternative: Load IDS dataset
# X, y, class_names = load_ids_dataset()

print(f"Loaded dataset: {X.shape[0]} samples, {X.shape[1]} features, {len(class_names)} classes")
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

# Initialize FL system with Dirichlet split
fl_system.initialize_fl_system(X, y, class_names, test_size=0.2, non_iid_factor=0.8)

# Show system readiness
print(f"\nSystem ready for FL training:")
print(f"  Clients: {len(fl_system.clients)}")
print(f"  Test dataset: {len(fl_system.test_dataset)} samples")
print(f"  Model architecture: {fl_system.fl_config}")

# Run a single FL round for each policy
test_policies = ["Random", "Greedy_composite", "MOO"]
for policy in test_policies:
    print(f"\n--- Testing {policy} Policy ---")
    round_result = fl_system.run_federated_round(policy, 3, 0)
    
    if 'error' not in round_result:
        global_metrics = round_result['global_metrics']
        print(f"Success! Global metrics:")
        print(f"  Accuracy: {global_metrics['accuracy']:.4f}")
        print(f"  F1-macro: {global_metrics['f1_macro']:.4f}")
        print(f"  Precision: {global_metrics['precision_macro']:.4f}")
        print(f"  Recall: {global_metrics['recall_macro']:.4f}")
        print(f"  Energy: {round_result['total_energy']:.1f} mAh")

# Run mini experiment (3 rounds)
print(f"\n=== Running Mini FL Experiment ===")
experiment_results = fl_system.run_fl_experiment(["Random", "MOO"], num_rounds=3, clients_per_round=3)

# Compare results
print(f"\n=== Mini Experiment Results ===")
for policy, results in experiment_results.items():
    if results['final_metrics']:
        fm = results['final_metrics']
        print(f"\n{policy}:")
        print(f"  Accuracy: {fm['final_accuracy']:.4f}")
        print(f"  F1-macro: {fm['final_f1_macro']:.4f}")
        print(f"  Energy: {fm['total_energy']:.1f} mAh")
        print(f"  Carbon: {fm['total_carbon']:.2f}g CO2eq")
        print(f"  Network: {fm['final_network_lifetime']*100:.1f}% alive")

print(f"\nFL Model and Aggregation System integration complete!")

=== Testing FL Model and Aggregation Integration ===
Loading Fashion-MNIST dataset...
Loaded dataset: 70000 samples, 784 features, 10 classes
Class distribution: {0: 7000, 1: 7000, 2: 7000, 3: 7000, 4: 7000, 5: 7000, 6: 7000, 7: 7000, 8: 7000, 9: 7000}
Creating Dirichlet split with alpha=2.00
Client 0 (sensor): 5977 samples
  Top classes: {'Pullover': 1355, 'Trouser': 886, 'T-shirt/top': 812}
  Data diversity: 0.916 (1.0 = perfectly balanced)
Client 1 (sensor): 6148 samples
  Top classes: {'Trouser': 1734, 'Dress': 812, 'Ankle boot': 676}
  Data diversity: 0.922 (1.0 = perfectly balanced)
Client 2 (sensor): 6029 samples
  Top classes: {'Dress': 2302, 'Pullover': 1027, 'Sneaker': 456}
  Data diversity: 0.840 (1.0 = perfectly balanced)
Client 3 (gateway): 5976 samples
  Top classes: {'Coat': 1451, 'Trouser': 895, 'T-shirt/top': 861}
  Data diversity: 0.921 (1.0 = perfectly balanced)
Client 4 (gateway): 6050 samples
  Top classes: {'Pullover': 1546, 'T-shirt/top': 850, 'Trouser': 829}
  D